# SIMCA peanut detection : clean calibration / validation / test workflow

- 2 calibration batches for fitting the one-class peanut SIMCA model;
- 1 validation batch for hyperparameter orientation;
- 1 final test batch, used only after selecting the configuration;
- empirical decision thresholds fixed at the 95% empirical quantile of the cross-validated target-class distribution;
- ranking focused on minimizing false negatives first, then false positives, with accuracy and F1 score as additional comparison metrics.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database_h5 import load_nir_uco_h5
from src.pixel_projection import (
    add_pixel_truth_labels,
    object_threshold_grid,
    binary_detection_metrics,
    plot_pixel_error_overlay,
    plot_pixel_fp_fn_overlay,
)
from src.simca_cv_calibration import (
    calibrate_simca_thresholds_cv,
    fit_final_simca_model,
    project_pixels_with_rule_variants,
    summarize_cv_calibration,
    run_simca_rule_variant_grid,
)
from src.border_decision import (
    aggregate_pixel_predictions_to_objects_core,
    border_width_object_threshold_grid,
    summarize_pixel_errors_by_border_zone,
)

pd.set_option("display.max_columns", 160)
RANDOM_STATE = 42

## 1. Load database

In [2]:
H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_all_images_reflectance.h5"
)

object_db, image_db = load_nir_uco_h5(
    H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

wavelengths = np.linspace(889, 1702, 69)[6:]

print("n objects:", len(object_db))
print("n images:", len(image_db))

n objects: 1262
n images: 48


## 2. Experimental split

The split below is deliberately strict:

- calibration: peanut pure objects from batches 1 and 2;
- validation: pure almond + pure peanut objects from batch 3;
- test: pure almond + pure peanut objects from batch 4.

The validation batch is used for hyperparameter orientation. The test batch is not used until the final evaluation.

In [3]:
CALIBRATION_BATCHES = [1, 2]
VALIDATION_BATCH = 3
TEST_BATCH = 4

TRAIN_FILTERS_CALIBRATION = {
    "sample_kind": ["pure"],
    "object_nut_type": ["peanut"],
    "batch": CALIBRATION_BATCHES,
}

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": [VALIDATION_BATCH],
}

TEST_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": ["almond", "peanut"],
    "batch": [TEST_BATCH],
}

PROJECTION_FILTERS_MIXTURES = {
    "sample_kind": ["mixture"],
}

## 2bis. Experimental protocol lock

The experimental protocol is fixed before final interpretation.

- Calibration uses only pure peanut objects from batches 1 and 2.
- Validation uses pure almond and pure peanut objects from batch 3.
- Test uses pure almond and pure peanut objects from batch 4.
- Mixtures are not used for hyperparameter selection.
- Mixtures are used only as an application/projection set.
- For deployment on mixtures, the selected configuration may be refitted on all pure peanut batches 1–4, but the selected hyperparameters must remain fixed.

In [5]:
FIXED_ALPHA = 0.05
EXPERIMENTAL_PROTOCOL = {
    "calibration_batches": CALIBRATION_BATCHES,
    "validation_batch": VALIDATION_BATCH,
    "test_batch": TEST_BATCH,
    "mixture_role": "projection_only",
    "empirical_threshold_quantile": 1.0 - FIXED_ALPHA,
    "selection_priority": [
        "minimize_fn_rate",
        "minimize_fp_rate",
        "maximize_f1_score",
        "maximize_accuracy",
        "maximize_balanced_accuracy",
    ],
}

display(pd.DataFrame([EXPERIMENTAL_PROTOCOL]))


def object_db_summary_table(object_db):
    rows = []

    for object_id, obj in object_db.items():
        rows.append({
            "object_id": object_id,
            "source_image": obj.get("source_clean_key"),
            "sample_kind": obj.get("sample_kind"),
            "object_nut_type": obj.get("object_nut_type"),
            "batch": obj.get("batch"),
            "area_pixels": obj.get("area_pixels"),
        })

    return pd.DataFrame(rows)


object_summary_df = object_db_summary_table(object_db)

split_summary_df = (
    object_summary_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
    .sort_values(["sample_kind", "object_nut_type", "batch"])
)

display(split_summary_df)


# Explicit leakage checks for pure-object validation/test protocol
calibration_batches_set = set(CALIBRATION_BATCHES)
validation_batches_set = {VALIDATION_BATCH}
test_batches_set = {TEST_BATCH}

assert calibration_batches_set.isdisjoint(validation_batches_set), "Calibration and validation batches overlap."
assert calibration_batches_set.isdisjoint(test_batches_set), "Calibration and test batches overlap."
assert validation_batches_set.isdisjoint(test_batches_set), "Validation and test batches overlap."

print("Protocol locked: no batch overlap between calibration, validation and test.")

,calibration_batches,validation_batch,test_batch,mixture_role,empirical_threshold_quantile,selection_priority
0,"[1, 2]",3,4,projection_only,0.95,"[minimize_fn_rate, minimize_fp_rate, maximize_..."


,sample_kind,object_nut_type,batch,n_objects
0,mixture,unknown,NaN,722
1,position_reference,peanut,1.0,47
2,position_reference,peanut,2.0,47
3,position_reference,peanut,3.0,47
4,position_reference,peanut,4.0,5
5,pure,almond,1.0,52
6,pure,almond,2.0,59
7,pure,almond,3.0,55
8,pure,almond,4.0,48
9,pure,peanut,1.0,46


Protocol locked: no batch overlap between calibration, validation and test.


## 3. Metrics and hierarchical ranking

The selection objective is not “best possible result”; it is an orientation of hyperparameters. We therefore use a transparent hierarchy:

1. minimize false-negative rate (`fn_rate`);
2. then minimize false-positive rate (`fp_rate`);
3. then maximize F1 score;
4. then maximize accuracy and balanced accuracy.

For scalar scores and Optuna-like ranking, the recommended cost is:

$$
\mathrm{cost} = 10 	\times \mathrm{FN\ rate} + 1 	\times \mathrm{FP\ rate}
$$

The 10:1 ratio makes false negatives the dominant criterion, without completely ignoring false positives.

In [6]:
def add_orientation_columns(df):
    out = df.copy()

    if "fn_rate" not in out.columns and "peanut_sensitivity" in out.columns:
        out["fn_rate"] = 1.0 - out["peanut_sensitivity"]
    if "fp_rate" not in out.columns and "almond_specificity" in out.columns:
        out["fp_rate"] = 1.0 - out["almond_specificity"]

    for col in ["f1_score", "accuracy", "balanced_accuracy"]:
        if col not in out.columns:
            out[col] = np.nan

    # Cost used only as an orientation score, not as the primary ranking.
    out["fn_fp_cost_10_1"] = 10.0 * out["fn_rate"] + out["fp_rate"]
    out["orientation_score"] = (
        -out["fn_fp_cost_10_1"]
        + 0.05 * out["f1_score"].fillna(0)
        + 0.02 * out["accuracy"].fillna(0)
    )
    return out


def rank_simca_results(df):
    """
    Strict hierarchy:
    1. minimize FN rate;
    2. minimize FP rate;
    3. maximize F1;
    4. maximize accuracy;
    5. maximize balanced accuracy;
    6. maximize orientation score.
    """
    out = add_orientation_columns(df)

    return (
        out.sort_values(
            [
                "fn_rate",
                "fp_rate",
                "f1_score",
                "accuracy",
                "balanced_accuracy",
                "orientation_score",
            ],
            ascending=[True, True, False, False, False, False],
        )
        .reset_index(drop=True)
    )


DISPLAY_COLS = [
    "preprocessing", "n_components", "rule_variant", "object_threshold",
    "tp", "fn", "fp", "tn", "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity", "accuracy", "f1_score", "balanced_accuracy",
    "orientation_score", "cv_target_rejection_rate", "cv_rule_limit",
]

## 4. Hyperparameter grid on validation batch only

For empirical decision rules, `alpha=0.05` means that thresholds are the 95% quantile of the empirical cross-validated target-class statistics.

Recommended search space:

- preprocessing: keep a small, interpretable set;
- number of PCA components: local range around plausible values;
- rule variant: compare theoretical references and empirical 95% rules;
- object threshold: fixed here first, then refined separately using the validation batch.

In [7]:
SEARCH_PREPROCESSING_CONFIGS = {
    "raw": (),
    "absorbance": ("absorbance",),
    "snv": ("snv",),
    "absorbance_snv": ("absorbance", "snv"),
    "absorbance_sg_d1": ("absorbance", "sg_d1"),
    "absorbance_snv_sg_d1": ("absorbance", "snv", "sg_d1"),
}

RULE_VARIANTS_FOR_SELECTION = [
    # theoretical references
    "simple_chi2",
    "alternative_chi2_fixed2",
    "data_driven_chi2",
    # empirical 95% CV rules
    "simple_emp_cv",
    "alternative_chi2_emp_cv",
    "data_driven_emp_cv",
]

N_COMPONENTS_GRID = [4, 6, 7, 8, 9, 10, 11]

FIXED_MATRIX_METHOD = "balanced_pixels"
FIXED_ALPHA = 0.05          # empirical quantile = 95%
FIXED_M = 40
FIXED_OBJECT_THRESHOLD = 0.75
FIXED_SG_WINDOW_LENGTH = 11
FIXED_SG_POLYORDER = 2
FIXED_POSITION_DILATION_RADIUS = 3
CV_N_SPLITS = None          # None = Leave-One-Object-Out; use 5 for a faster GroupKFold

In [8]:
validation_grid_df, validation_grid_results, validation_grid_errors = run_simca_rule_variant_grid(
    object_db=object_db,
    image_db=image_db,
    train_filters=TRAIN_FILTERS_CALIBRATION,
    projection_filters=VALIDATION_FILTERS,
    preprocessing_configs=SEARCH_PREPROCESSING_CONFIGS,
    rule_variants=RULE_VARIANTS_FOR_SELECTION,
    n_components_values=N_COMPONENTS_GRID,
    matrix_method=FIXED_MATRIX_METHOD,
    alpha=FIXED_ALPHA,
    object_threshold=FIXED_OBJECT_THRESHOLD,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
    position_dilation_radius=FIXED_POSITION_DILATION_RADIUS,
    cv_n_splits=CV_N_SPLITS,
    group_col="object_id",
    keep_pixel_tables=False,
    keep_cv_tables=False,
    verbose=True,
)




[1/42] preprocessing=raw | A=4 | alpha=0.05

[2/42] preprocessing=raw | A=6 | alpha=0.05

[3/42] preprocessing=raw | A=7 | alpha=0.05

[4/42] preprocessing=raw | A=8 | alpha=0.05

[5/42] preprocessing=raw | A=9 | alpha=0.05

[6/42] preprocessing=raw | A=10 | alpha=0.05

[7/42] preprocessing=raw | A=11 | alpha=0.05

[8/42] preprocessing=absorbance | A=4 | alpha=0.05

[9/42] preprocessing=absorbance | A=6 | alpha=0.05

[10/42] preprocessing=absorbance | A=7 | alpha=0.05

[11/42] preprocessing=absorbance | A=8 | alpha=0.05

[12/42] preprocessing=absorbance | A=9 | alpha=0.05

[13/42] preprocessing=absorbance | A=10 | alpha=0.05

[14/42] preprocessing=absorbance | A=11 | alpha=0.05

[15/42] preprocessing=snv | A=4 | alpha=0.05

[16/42] preprocessing=snv | A=6 | alpha=0.05

[17/42] preprocessing=snv | A=7 | alpha=0.05

[18/42] preprocessing=snv | A=8 | alpha=0.05

[19/42] preprocessing=snv | A=9 | alpha=0.05

[20/42] preprocessing=snv | A=10 | alpha=0.05

[21/42] preprocessing=snv | A=11 |

In [9]:
validation_grid_ranked = rank_simca_results(validation_grid_df)

print("Errors:")
display(validation_grid_errors)

available_cols = [c for c in DISPLAY_COLS if c in validation_grid_ranked.columns]
display(validation_grid_ranked[available_cols].head(30))

Errors:


""


,preprocessing,n_components,rule_variant,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,orientation_score,cv_target_rejection_rate,cv_rule_limit
0,absorbance_sg_d1,6,simple_emp_cv,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,-0.117980,0.050092,2.082295
1,absorbance_sg_d1,6,data_driven_emp_cv,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,-0.117980,0.050092,158.208911
2,absorbance_sg_d1,11,simple_emp_cv,0.75,53.0,0.0,12.0,43.0,0.0,0.218182,1.0,0.781818,0.888889,0.898305,0.890909,-0.155489,0.050092,1.797677
3,absorbance_sg_d1,11,data_driven_emp_cv,0.75,53.0,0.0,12.0,43.0,0.0,0.218182,1.0,0.781818,0.888889,0.898305,0.890909,-0.155489,0.050092,136.654622
4,absorbance_sg_d1,6,alternative_chi2_emp_cv,0.75,53.0,0.0,15.0,40.0,0.0,0.272727,1.0,0.727273,0.861111,0.876033,0.863636,-0.211703,0.050092,2.813204
5,absorbance_sg_d1,10,alternative_chi2_emp_cv,0.75,53.0,0.0,15.0,40.0,0.0,0.272727,1.0,0.727273,0.861111,0.876033,0.863636,-0.211703,0.050092,2.692587
6,absorbance_sg_d1,10,simple_emp_cv,0.75,53.0,0.0,17.0,38.0,0.0,0.309091,1.0,0.690909,0.842593,0.861789,0.845455,-0.249150,0.050092,1.842042
7,absorbance_sg_d1,10,data_driven_emp_cv,0.75,53.0,0.0,17.0,38.0,0.0,0.309091,1.0,0.690909,0.842593,0.861789,0.845455,-0.249150,0.050092,136.135514
8,absorbance_sg_d1,9,alternative_chi2_emp_cv,0.75,53.0,0.0,19.0,36.0,0.0,0.345455,1.0,0.654545,0.824074,0.848000,0.827273,-0.286573,0.050092,2.728386
9,absorbance_sg_d1,9,simple_emp_cv,0.75,53.0,0.0,22.0,33.0,0.0,0.400000,1.0,0.600000,0.796296,0.828125,0.800000,-0.342668,0.050092,1.869906


## 5. Summaries: orientation rather than final performance

These tables answer: “which hyperparameters tend to reduce FN first, then FP?” They should not be presented as final performance estimates.

In [10]:
def summarize_orientation(df, group_cols):
    d = add_orientation_columns(df)
    summary = (
        d.groupby(group_cols, as_index=False)
        .agg(
            mean_fn_rate=("fn_rate", "mean"),
            mean_fp_rate=("fp_rate", "mean"),
            mean_f1=("f1_score", "mean"),
            mean_accuracy=("accuracy", "mean"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            mean_orientation_score=("orientation_score", "mean"),
            n_configs=("orientation_score", "size"),
        )
    )
    return summary.sort_values(
        ["mean_fn_rate", "mean_fp_rate", "mean_f1", "mean_accuracy"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)

preproc_orientation = summarize_orientation(validation_grid_ranked, ["preprocessing"])
component_orientation = summarize_orientation(validation_grid_ranked, ["n_components"])
rule_orientation = summarize_orientation(validation_grid_ranked, ["rule_variant"])

print("Preprocessing orientation")
display(preproc_orientation)
print("Number of components orientation")
display(component_orientation)
print("Decision rule orientation")
display(rule_orientation)

Preprocessing orientation


,preprocessing,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,snv,0.039533,0.847186,0.679018,0.549162,0.556641,-1.197580,42
1,absorbance_snv,0.048068,0.799567,0.689861,0.569224,0.576182,-1.234372,42
2,absorbance_snv_sg_d1,0.062444,0.628139,0.733994,0.649471,0.654709,-1.202888,42
3,raw,0.076370,0.532900,0.752146,0.691138,0.695365,-1.245172,42
4,absorbance,0.093441,0.482251,0.751848,0.708554,0.712154,-1.364899,42
5,absorbance_sg_d1,0.222372,0.177489,0.769379,0.800485,0.800069,-2.346730,42


Number of components orientation


,n_components,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,4,0.063417,0.798485,0.679709,0.562243,0.569049,-1.387426,36
1,6,0.067610,0.603535,0.735112,0.659465,0.664427,-1.229691,36
2,7,0.075996,0.542424,0.750711,0.686471,0.690790,-1.251117,36
3,8,0.092243,0.551515,0.735963,0.673868,0.678121,-1.423672,36
4,9,0.094340,0.537374,0.737441,0.680041,0.684143,-1.430297,36
5,10,0.113732,0.514141,0.735081,0.682356,0.686063,-1.601057,36
6,11,0.125262,0.497980,0.731606,0.684928,0.688379,-1.700322,36


Decision rule orientation


,rule_variant,mean_fn_rate,mean_fp_rate,mean_f1,mean_accuracy,mean_balanced_accuracy,mean_orientation_score,n_configs
0,data_driven_emp_cv,0.009883,0.727706,0.729460,0.624559,0.631206,-0.777573,42
1,alternative_chi2_emp_cv,0.009883,0.758874,0.719606,0.608686,0.615621,-0.809552,42
2,simple_emp_cv,0.012579,0.738095,0.724935,0.617945,0.624663,-0.815276,42
3,alternative_chi2_fixed2,0.065139,0.584848,0.745379,0.670194,0.675006,-1.185568,42
4,data_driven_chi2,0.184636,0.380087,0.737354,0.715829,0.717639,-2.175264,42
5,simple_chi2,0.260108,0.277922,0.719514,0.730820,0.730985,-2.828408,42


In [11]:
fig = px.scatter(
    validation_grid_ranked,
    x="fp_rate",
    y="fn_rate",
    color="rule_variant",
    symbol="preprocessing",
    hover_data=["n_components", "accuracy", "f1_score", "balanced_accuracy", "tp", "fn", "fp", "tn"],
    title="Validation orientation: FN rate first, then FP rate",
)
fig.update_layout(xaxis_title="False-positive rate", yaxis_title="False-negative rate")
fig.show()

## 6. Select spectral SIMCA configuration on validation

The selected row is the first row after hierarchical ranking. You can override it manually, but do not inspect the test batch while doing so.

In [12]:
SELECTED_ROW = validation_grid_ranked.iloc[0].copy()

SELECTED_PREPROCESSING = str(SELECTED_ROW["preprocessing"])
SELECTED_PREPROCESSING_STEPS = SEARCH_PREPROCESSING_CONFIGS[SELECTED_PREPROCESSING]
SELECTED_N_COMPONENTS = int(SELECTED_ROW["n_components"])
SELECTED_RULE_VARIANT = str(SELECTED_ROW["rule_variant"])
SELECTED_ALPHA = FIXED_ALPHA

print("Selected spectral configuration from validation:")
print("preprocessing:", SELECTED_PREPROCESSING, SELECTED_PREPROCESSING_STEPS)
print("n_components:", SELECTED_N_COMPONENTS)
print("rule_variant:", SELECTED_RULE_VARIANT)
print("alpha / empirical quantile:", SELECTED_ALPHA, "/", 1 - SELECTED_ALPHA)

display(pd.DataFrame([SELECTED_ROW])[available_cols])

Selected spectral configuration from validation:
preprocessing: absorbance_sg_d1 ('absorbance', 'sg_d1')
n_components: 6
rule_variant: simple_emp_cv
alpha / empirical quantile: 0.05 / 0.95


,preprocessing,n_components,rule_variant,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,orientation_score,cv_target_rejection_rate,cv_rule_limit
0,absorbance_sg_d1,6,simple_emp_cv,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,-0.11798,0.050092,2.082295


## 7. Fit selected model and project validation/test

The model is still trained only on the two calibration batches. The validation batch is used for decision refinement; the test batch is only evaluated after the decision parameters are frozen.

In [13]:
def fit_project_selected(projection_filters, projection_name):
    cv_df, cv_thresholds = calibrate_simca_thresholds_cv(
        object_db=object_db,
        train_filters=TRAIN_FILTERS_CALIBRATION,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
        n_components=SELECTED_N_COMPONENTS,
        alpha=SELECTED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
        group_col="object_id",
        n_splits=CV_N_SPLITS,
    )
    cv_summary = summarize_cv_calibration(cv_df, cv_thresholds)

    final_bundle = fit_final_simca_model(
        object_db=object_db,
        train_filters=TRAIN_FILTERS_CALIBRATION,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
        n_components=SELECTED_N_COMPONENTS,
        alpha=SELECTED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
    )

    pixel_wide_df, simca_values, X_pixel = project_pixels_with_rule_variants(
        object_db=object_db,
        final_bundle=final_bundle,
        projection_filters=projection_filters,
        cv_thresholds=cv_thresholds,
        rule_variants=[SELECTED_RULE_VARIANT],
    )

    pixel_wide_df = add_pixel_truth_labels(
        pixel_df=pixel_wide_df,
        image_db=image_db,
        object_db=object_db,
        dilation_radius=FIXED_POSITION_DILATION_RADIUS,
    )

    pixel_df = pixel_wide_df.copy()
    rule = SELECTED_RULE_VARIANT
    pixel_df["predicted_peanut_pixel"] = pixel_df[f"pred_{rule}"].astype(bool)
    pixel_df["predicted_label_pixel"] = np.where(pixel_df["predicted_peanut_pixel"], "peanut", "non_peanut")
    pixel_df["rule_statistic"] = pixel_df[f"stat_{rule}"]
    pixel_df["rule_limit"] = pixel_df[f"limit_{rule}"]
    pixel_df["rule_variant"] = rule
    pixel_df["projection_name"] = projection_name

    return {
        "projection_name": projection_name,
        "cv_df": cv_df,
        "cv_thresholds": cv_thresholds,
        "cv_summary": cv_summary,
        "final_bundle": final_bundle,
        "pixel_df": pixel_df,
        "simca_values": simca_values,
        "X_pixel": X_pixel,
    }

validation_selected = fit_project_selected(VALIDATION_FILTERS, "validation")
test_selected = fit_project_selected(TEST_FILTERS, "test")

print("CV calibration summary for the selected configuration:")
display(validation_selected["cv_summary"])

CV calibration summary for the selected configuration:


,rule_variant,stat_col,limit,n,n_rejected_target_cv,rejection_rate_target_cv,acceptance_rate_target_cv,expected_rejection_rate,expected_acceptance_rate,abs_rejection_error
0,simple_emp_cv,simple_chi2_stat,2.082295,3813,191,0.050092,0.949908,0.05,0.95,0.000092
1,alternative_chi2_emp_cv,alternative_chi2_stat,2.813204,3813,191,0.050092,0.949908,0.05,0.95,0.000092
2,data_driven_emp_cv,data_driven_stat,158.208911,3813,191,0.050092,0.949908,0.05,0.95,0.000092
3,alternative_empHQ_emp_cv,alternative_empHQ_stat,1.783779,3813,191,0.050092,0.949908,0.05,0.95,0.000092
4,alternative_chi2_fixed2,alternative_chi2_stat,2.000000,3813,433,0.113559,0.886441,0.05,0.95,0.063559
5,simple_chi2,simple_chi2_stat,1.000000,3813,897,0.235248,0.764752,0.05,0.95,0.185248


## 8. Refine object decision on validation only

This is where we tune the object-level decision threshold and optional border exclusion. This step is not a spectral SIMCA hyperparameter; it is a decision aggregation parameter.

In [14]:
BORDER_WIDTHS = [0, 1, 2, 3]
OBJECT_THRESHOLDS_FOR_DECISION = np.round(np.arange(0.60, 0.96, 0.05), 2)
MIN_CORE_PIXELS = 20

validation_decision_df, validation_decision_tables = border_width_object_threshold_grid(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    border_widths=BORDER_WIDTHS,
    object_thresholds=OBJECT_THRESHOLDS_FOR_DECISION,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

validation_decision_ranked = rank_simca_results(validation_decision_df)

cols_decision = [
    "border_width", "object_threshold", "tp", "fn", "fp", "tn",
    "fn_rate", "fp_rate", "peanut_sensitivity", "almond_specificity",
    "accuracy", "f1_score", "balanced_accuracy", "mean_core_fraction", "fallback_object_rate", "orientation_score",
]
display(validation_decision_ranked[cols_decision].head(30))

,border_width,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,mean_core_fraction,fallback_object_rate,orientation_score
0,0,0.75,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,1.000000,0.009259,-0.117980
1,2,0.75,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,0.391751,0.388889,-0.117980
2,1,0.75,53,0,11,44,0.000000,0.200000,1.000000,0.800000,0.898148,0.905983,0.900000,0.688626,0.083333,-0.136738
3,3,0.75,53,0,11,44,0.000000,0.200000,1.000000,0.800000,0.898148,0.905983,0.900000,0.117780,0.861111,-0.136738
4,0,0.70,53,0,12,43,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,1.000000,0.009259,-0.155489
5,1,0.70,53,0,12,43,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,0.688626,0.083333,-0.155489
6,2,0.65,53,0,12,43,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,0.391751,0.388889,-0.155489
7,2,0.70,53,0,12,43,0.000000,0.218182,1.000000,0.781818,0.888889,0.898305,0.890909,0.391751,0.388889,-0.155489
8,0,0.65,53,0,13,42,0.000000,0.236364,1.000000,0.763636,0.879630,0.890756,0.881818,1.000000,0.009259,-0.174233
9,3,0.70,53,0,13,42,0.000000,0.236364,1.000000,0.763636,0.879630,0.890756,0.881818,0.117780,0.861111,-0.174233


In [15]:
SELECTED_DECISION_ROW = validation_decision_ranked.iloc[0].copy()
SELECTED_BORDER_WIDTH = int(SELECTED_DECISION_ROW["border_width"])
SELECTED_OBJECT_THRESHOLD = float(SELECTED_DECISION_ROW["object_threshold"])

print("Selected object-decision parameters from validation:")
print("border_width:", SELECTED_BORDER_WIDTH)
print("object_threshold:", SELECTED_OBJECT_THRESHOLD)
display(pd.DataFrame([SELECTED_DECISION_ROW])[cols_decision])

Selected object-decision parameters from validation:
border_width: 0
object_threshold: 0.75


,border_width,object_threshold,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,mean_core_fraction,fallback_object_rate,orientation_score
0,0.0,0.75,53.0,0.0,10.0,45.0,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,1.0,0.009259,-0.11798


## 9. Final test evaluation

The following cell applies the selected spectral configuration and selected object-decision parameters to the test batch. Do not modify the chosen parameters based on this output.

In [16]:
validation_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

test_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=test_selected["pixel_df"],
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

validation_metrics = binary_detection_metrics(
    validation_object_df,
    true_col="true_peanut_object",
    pred_col="predicted_peanut_object",
)
validation_metrics["set"] = "validation"

test_metrics = binary_detection_metrics(
    test_object_df,
    true_col="true_peanut_object",
    pred_col="predicted_peanut_object",
)
test_metrics["set"] = "test"

final_comparison_df = add_orientation_columns(pd.DataFrame([validation_metrics, test_metrics]))
final_comparison_df["preprocessing"] = SELECTED_PREPROCESSING
final_comparison_df["n_components"] = SELECTED_N_COMPONENTS
final_comparison_df["rule_variant"] = SELECTED_RULE_VARIANT
final_comparison_df["border_width"] = SELECTED_BORDER_WIDTH
final_comparison_df["object_threshold"] = SELECTED_OBJECT_THRESHOLD

display(final_comparison_df[[
    "set", "tp", "fn", "fp", "tn", "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity", "accuracy", "f1_score", "balanced_accuracy",
    "preprocessing", "n_components", "rule_variant", "border_width", "object_threshold",
]])

,set,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold
0,validation,53,0,10,45,0.0,0.181818,1.0,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,simple_emp_cv,0,0.75
1,test,29,0,15,33,0.0,0.312500,1.0,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,simple_emp_cv,0,0.75


## 9bis. Clean comparison of selected baseline models

This section compares a small set of interpretable baseline models.

The comparison is done in two steps:

1. validation batch 3 is used to compare and orient the models;
2. test batch 4 is used only as an external evaluation.

The mixtures are not used to select the model. They are analyzed later as an application/projection set.

In [17]:
candidate_models = pd.DataFrame([
    {
        "model_name": "A_safety_FN0",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.75,
        "selection_logic": "FN=0, then min FP",
    },
    {
        "model_name": "B_compromise",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.80,
        "selection_logic": "cost compromise",
    },
    {
        "model_name": "C_accuracy",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.85,
        "selection_logic": "max accuracy / F1",
    },
    {
        "model_name": "D_simple_ablation",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 6,
        "rule_variant": "simple_emp_cv",
        "border_width": 0,
        "object_threshold": 0.75,
        "selection_logic": "simple empirical SIMCA ablation",
    },
    {
        "model_name": "E_dd_ablation",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 6,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 0,
        "object_threshold": 0.75,
        "selection_logic": "DD empirical SIMCA ablation",
    },
])

display(candidate_models)

,model_name,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
0,A_safety_FN0,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,"FN=0, then min FP"
1,B_compromise,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,cost compromise
2,C_accuracy,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1
3,D_simple_ablation,absorbance_sg_d1,6,simple_emp_cv,0,0.75,simple empirical SIMCA ablation
4,E_dd_ablation,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75,DD empirical SIMCA ablation


In [18]:
candidate_projection_cache = {}


def _filters_to_key(filters):
    """Make filter dictionaries hashable for caching."""
    items = []
    for key, value in sorted(filters.items()):
        if isinstance(value, (list, tuple, set, np.ndarray)):
            value_key = tuple(value)
        else:
            value_key = value
        items.append((key, value_key))
    return tuple(items)


def get_candidate_projection(
    candidate_row,
    projection_filters,
    projection_name,
    train_filters=TRAIN_FILTERS_CALIBRATION,
):
    """
    Fit one candidate SIMCA model on calibration batches,
    project one target set, and return pixel-level predictions.

    Results are cached because several candidates share the same spectral model
    and only differ by object_threshold.
    """
    preprocessing = str(candidate_row["preprocessing"])
    preprocessing_steps = tuple(SEARCH_PREPROCESSING_CONFIGS[preprocessing])
    n_components = int(candidate_row["n_components"])
    rule_variant = str(candidate_row["rule_variant"])

    cache_key = (
        _filters_to_key(train_filters),
        _filters_to_key(projection_filters),
        str(projection_name),
        preprocessing,
        preprocessing_steps,
        n_components,
        rule_variant,
        float(FIXED_ALPHA),
        int(FIXED_M),
        int(FIXED_SG_WINDOW_LENGTH),
        int(FIXED_SG_POLYORDER),
    )

    if cache_key in candidate_projection_cache:
        return candidate_projection_cache[cache_key]

    cv_df, cv_thresholds = calibrate_simca_thresholds_cv(
        object_db=object_db,
        train_filters=train_filters,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=preprocessing_steps,
        n_components=n_components,
        alpha=FIXED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
        group_col="object_id",
        n_splits=CV_N_SPLITS,
    )

    final_bundle = fit_final_simca_model(
        object_db=object_db,
        train_filters=train_filters,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=preprocessing_steps,
        n_components=n_components,
        alpha=FIXED_ALPHA,
        m=FIXED_M,
        random_state=RANDOM_STATE,
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
    )

    pixel_wide_df, simca_values, X_pixel = project_pixels_with_rule_variants(
        object_db=object_db,
        final_bundle=final_bundle,
        projection_filters=projection_filters,
        cv_thresholds=cv_thresholds,
        rule_variants=[rule_variant],
    )

    pixel_wide_df = add_pixel_truth_labels(
        pixel_df=pixel_wide_df,
        image_db=image_db,
        object_db=object_db,
        dilation_radius=FIXED_POSITION_DILATION_RADIUS,
    )

    pixel_df = pixel_wide_df.copy()
    pixel_df["predicted_peanut_pixel"] = pixel_df[f"pred_{rule_variant}"].astype(bool)
    pixel_df["predicted_label_pixel"] = np.where(
        pixel_df["predicted_peanut_pixel"],
        "peanut",
        "non_peanut",
    )
    pixel_df["rule_statistic"] = pixel_df[f"stat_{rule_variant}"]
    pixel_df["rule_limit"] = pixel_df[f"limit_{rule_variant}"]
    pixel_df["rule_variant"] = rule_variant
    pixel_df["projection_name"] = projection_name

    out = {
        "projection_name": projection_name,
        "candidate_row": candidate_row.copy(),
        "cv_df": cv_df,
        "cv_thresholds": cv_thresholds,
        "final_bundle": final_bundle,
        "pixel_df": pixel_df,
        "simca_values": simca_values,
        "X_pixel": X_pixel,
    }

    candidate_projection_cache[cache_key] = out
    return out


def evaluate_candidate_hard_decision(
    candidate_row,
    projection_filters,
    projection_name,
    train_filters=TRAIN_FILTERS_CALIBRATION,
):
    """
    Evaluate one candidate using its fixed hard object decision:
        peanut_pixel_ratio >= object_threshold
    """
    projection = get_candidate_projection(
        candidate_row=candidate_row,
        projection_filters=projection_filters,
        projection_name=projection_name,
        train_filters=train_filters,
    )

    object_df = aggregate_pixel_predictions_to_objects_core(
        pixel_df=projection["pixel_df"],
        object_db=object_db,
        object_threshold=float(candidate_row["object_threshold"]),
        border_width=int(candidate_row["border_width"]),
        min_core_pixels=MIN_CORE_PIXELS,
        fallback_to_all_pixels=True,
    )

    metrics = binary_detection_metrics(
        object_df,
        true_col="true_peanut_object",
        pred_col="predicted_peanut_object",
    )

    metrics.update({
        "model_name": str(candidate_row["model_name"]),
        "decision_model": "hard_ratio_threshold",
        "set": projection_name,
        "preprocessing": str(candidate_row["preprocessing"]),
        "n_components": int(candidate_row["n_components"]),
        "rule_variant": str(candidate_row["rule_variant"]),
        "border_width": int(candidate_row["border_width"]),
        "object_threshold": float(candidate_row["object_threshold"]),
        "selection_logic": str(candidate_row["selection_logic"]),
    })

    return metrics, object_df, projection

In [19]:
hard_comparison_rows = []
candidate_outputs = {}

for _, row in candidate_models.iterrows():
    model_name = str(row["model_name"])

    for projection_name, filters in [
        ("validation", VALIDATION_FILTERS),
        ("test", TEST_FILTERS),
    ]:
        metrics, object_df, projection = evaluate_candidate_hard_decision(
            candidate_row=row,
            projection_filters=filters,
            projection_name=projection_name,
            train_filters=TRAIN_FILTERS_CALIBRATION,
        )

        hard_comparison_rows.append(metrics)

        candidate_outputs[(model_name, projection_name)] = {
            "metrics": metrics,
            "object_df": object_df,
            "pixel_df": projection["pixel_df"],
            "projection": projection,
        }

hard_model_comparison_df = add_orientation_columns(
    pd.DataFrame(hard_comparison_rows)
)

hard_cols = [
    "model_name",
    "decision_model",
    "set",
    "tp", "fn", "fp", "tn",
    "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity",
    "accuracy", "f1_score", "balanced_accuracy",
    "preprocessing", "n_components", "rule_variant",
    "border_width", "object_threshold",
    "selection_logic",
]

display(
    hard_model_comparison_df[hard_cols]
    .sort_values(["set", "fn_rate", "fp_rate", "f1_score", "accuracy"],
                 ascending=[True, True, True, False, False])
    .reset_index(drop=True)
)

,model_name,decision_model,set,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
0,C_accuracy,hard_ratio_threshold,test,29,0,6,42,0.000000,0.125000,1.000000,0.875000,0.922078,0.906250,0.937500,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1
1,B_compromise,hard_ratio_threshold,test,29,0,7,41,0.000000,0.145833,1.000000,0.854167,0.909091,0.892308,0.927083,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,cost compromise
2,A_safety_FN0,hard_ratio_threshold,test,29,0,9,39,0.000000,0.187500,1.000000,0.812500,0.883117,0.865672,0.906250,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,"FN=0, then min FP"
3,D_simple_ablation,hard_ratio_threshold,test,29,0,15,33,0.000000,0.312500,1.000000,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,simple_emp_cv,0,0.75,simple empirical SIMCA ablation
4,E_dd_ablation,hard_ratio_threshold,test,29,0,15,33,0.000000,0.312500,1.000000,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75,DD empirical SIMCA ablation
5,A_safety_FN0,hard_ratio_threshold,validation,53,0,6,49,0.000000,0.109091,1.000000,0.890909,0.944444,0.946429,0.945455,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,"FN=0, then min FP"
6,D_simple_ablation,hard_ratio_threshold,validation,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,simple_emp_cv,0,0.75,simple empirical SIMCA ablation
7,E_dd_ablation,hard_ratio_threshold,validation,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75,DD empirical SIMCA ablation
8,B_compromise,hard_ratio_threshold,validation,52,1,4,51,0.018868,0.072727,0.981132,0.927273,0.953704,0.954128,0.954202,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,cost compromise
9,C_accuracy,hard_ratio_threshold,validation,51,2,2,53,0.037736,0.036364,0.962264,0.963636,0.962963,0.962264,0.962950,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1


In [20]:
hard_summary_pivot = (
    hard_model_comparison_df
    .pivot_table(
        index=[
            "model_name",
            "preprocessing",
            "n_components",
            "rule_variant",
            "border_width",
            "object_threshold",
            "selection_logic",
        ],
        columns="set",
        values=[
            "fn", "fp",
            "fn_rate", "fp_rate",
            "accuracy", "f1_score",
            "peanut_sensitivity", "almond_specificity",
            "balanced_accuracy",
        ],
        aggfunc="first",
    )
)

display(hard_summary_pivot)

accuracy  \
set                                                                                                                                   test   
model_name        preprocessing    n_components rule_variant       border_width object_threshold selection_logic                             
A_safety_FN0      absorbance_sg_d1 7            data_driven_emp_cv 1            0.75             FN=0, then min FP                0.883117   
B_compromise      absorbance_sg_d1 7            data_driven_emp_cv 1            0.80             cost compromise                  0.909091   
C_accuracy        absorbance_sg_d1 7            data_driven_emp_cv 1            0.85             max accuracy / F1                0.922078   
D_simple_ablation absorbance_sg_d1 6            simple_emp_cv      0            0.75             simple empirical SIMCA ablation  0.805195   
E_dd_ablation     absorbance_sg_d1 6            data_driven_emp_cv 0            0.75             DD empirical SIMCA ablation      0.805195   

                                                                                                                                             \
set                                                                                                                              validation   
model_name        preprocessing    n_components rule_variant       border_width object_threshold selection_logic                              
A_safety_FN0      absorbance_sg_d1 7            data_driven_emp_cv 1            0.75             FN=0, then min FP                 0.944444   
B_compromise      absorbance_sg_d1 7            data_driven_emp_cv 1            0.80             cost compromise                   0.953704   
C_accuracy        absorbance_sg_d1 7            data_driven_emp_cv 1            0.85             max accuracy / F1                 0.962963   
D_simple_ablation absorbance_sg_d1 6            simple_emp_cv      0            0.75             simple empirical SIMCA ablation   0.907407   
E_dd_ablation     absorbance_sg_d1 6            data_driven_emp_cv 0            0.75             DD empirical SIMCA ablation       0.907407   

                                                                                                                                 almond_specificity  \
set                                                                                                                                            test   
model_name        preprocessing    n_components rule_variant       border_width object_threshold selection_logic                                      
A_safety_FN0      absorbance_sg_d1 7            data_driven_emp_cv 1            0.75             FN=0, then min FP                         0.812500   
B_compromise      absorbance_sg_d1 7            data_driven_emp_cv 1            0.80             cost compromise                           0.854167   
C_accuracy        absorbance_sg_d1 7            data_driven_emp_cv 1            0.85             max accuracy / F1                         0.875000   
D_simple_ablation absorbance_sg_d1 6            simple_emp_cv      0            0.75             simple empirical SIMCA ablation           0.687500   
E_dd_ablation     absorbance_sg_d1 6            data_driven_emp_cv 0            0.75             DD empirical SIMCA ablation               0.687500   

                                                                                                                                             \
set                                                                                                                              validation   
model_name        preprocessing    n_components rule_variant       border_width object_threshold selection_logic                              
A_safety_FN0      absorbance_sg_d1 7            data_driven_emp_cv 1            0.75             FN=0, then min FP                 0.890909   
B_compromise      absorbance_sg_d1 7            data_driven_emp_cv 

## 9ter. Object-score modelling

The hard object decision uses a single threshold on the peanut pixel ratio.

Here, we model the object score more explicitly. The SIMCA pixel model remains unchanged. The new model only transforms pixel-level SIMCA outputs into an object-level peanut probability.

Important:
- the object-score model is fitted on the validation batch;
- the probability threshold is chosen on the validation batch;
- the test batch is used only for external evaluation.

In [21]:
from scipy.special import logit
from src.border_decision import add_border_flags_to_pixel_df


def aggregate_object_score_features(
    pixel_df,
    object_db,
    border_width=1,
    min_core_pixels=20,
    fallback_to_all_pixels=True,
    truth_threshold=0.50,
):
    """
    Build continuous object-level scores from pixel-level SIMCA outputs.

    Main features:
    - peanut_pixel_ratio
    - logit_peanut_ratio
    - summaries of normalized SIMCA statistic
    - summaries of SIMCA margin
    """
    df = pixel_df.copy()

    required_cols = [
        "predicted_peanut_pixel",
        "rule_statistic",
        "rule_limit",
        "object_id",
        "source_image",
        "row",
        "col",
    ]

    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in pixel_df: {missing}")

    df = add_border_flags_to_pixel_df(
        pixel_df=df,
        object_db=object_db,
        border_width=int(border_width),
        object_id_col="object_id",
        row_col="row",
        col_col="col",
    )

    df["predicted_peanut_pixel"] = df["predicted_peanut_pixel"].astype(bool)
    df["stat_norm"] = df["rule_statistic"].astype(float) / df["rule_limit"].astype(float)
    df["margin_norm"] = 1.0 - df["stat_norm"]

    rows = []

    for (object_id, source_image), group in df.groupby(["object_id", "source_image"], sort=False):
        core = group[group["is_core_pixel"]].copy()
        decision_group = core
        used_core = True

        if len(core) < int(min_core_pixels):
            if fallback_to_all_pixels:
                decision_group = group.copy()
                used_core = False
            else:
                decision_group = core.copy()

        if len(decision_group) == 0:
            continue

        n_total = int(len(group))
        n_decision = int(len(decision_group))
        n_core = int(group["is_core_pixel"].sum())
        n_border = int(group["is_border_pixel"].sum())

        k_peanut = int(decision_group["predicted_peanut_pixel"].sum())
        peanut_ratio = k_peanut / n_decision

        # Smoothed ratio avoids infinite logit values when ratio = 0 or 1.
        peanut_ratio_smooth = (k_peanut + 0.5) / (n_decision + 1.0)

        row = {
            "object_id": object_id,
            "source_image": source_image,

            "n_pixels_total": n_total,
            "n_pixels_decision": n_decision,
            "n_pixels_core": n_core,
            "n_pixels_border": n_border,
            "decision_used_core": bool(used_core),
            "border_width": int(border_width),

            "n_predicted_peanut_pixels": k_peanut,
            "peanut_pixel_ratio": float(peanut_ratio),
            "peanut_pixel_ratio_smooth": float(peanut_ratio_smooth),
            "logit_peanut_ratio": float(logit(peanut_ratio_smooth)),

            "stat_norm_mean": float(decision_group["stat_norm"].mean()),
            "stat_norm_median": float(decision_group["stat_norm"].median()),
            "stat_norm_q10": float(decision_group["stat_norm"].quantile(0.10)),
            "stat_norm_q90": float(decision_group["stat_norm"].quantile(0.90)),

            "margin_norm_mean": float(decision_group["margin_norm"].mean()),
            "margin_norm_median": float(decision_group["margin_norm"].median()),
            "margin_norm_q10": float(decision_group["margin_norm"].quantile(0.10)),
            "margin_norm_q90": float(decision_group["margin_norm"].quantile(0.90)),

            "H_mean": float(decision_group["H"].mean()) if "H" in decision_group.columns else np.nan,
            "Q_mean": float(decision_group["Q"].mean()) if "Q" in decision_group.columns else np.nan,
        }

        if "true_peanut_pixel" in group.columns:
            true_ratio_total = float(group["true_peanut_pixel"].astype(bool).mean())
            row["true_peanut_pixel_ratio_total"] = true_ratio_total
            row["true_peanut_object"] = bool(true_ratio_total >= truth_threshold)

        obj = object_db.get(str(object_id), {})
        row["area_pixels"] = obj.get("area_pixels", np.nan)
        row["batch"] = obj.get("batch", None)
        row["sample_kind"] = obj.get("sample_kind", None)
        row["object_nut_type"] = obj.get("object_nut_type", None)

        rows.append(row)

    return pd.DataFrame(rows)

In [22]:
OBJECT_SCORE_BASE_MODEL_NAME = "C_accuracy"

object_score_base_row = (
    candidate_models
    .loc[candidate_models["model_name"].eq(OBJECT_SCORE_BASE_MODEL_NAME)]
    .iloc[0]
    .copy()
)

print("Object-score base spectral model:")
display(pd.DataFrame([object_score_base_row]))

Object-score base spectral model:


,model_name,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
2,C_accuracy,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1


In [23]:
validation_score_df = aggregate_object_score_features(
    pixel_df=candidate_outputs[(OBJECT_SCORE_BASE_MODEL_NAME, "validation")]["pixel_df"],
    object_db=object_db,
    border_width=int(object_score_base_row["border_width"]),
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

test_score_df = aggregate_object_score_features(
    pixel_df=candidate_outputs[(OBJECT_SCORE_BASE_MODEL_NAME, "test")]["pixel_df"],
    object_db=object_db,
    border_width=int(object_score_base_row["border_width"]),
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

print("Validation object-score table:")
display(validation_score_df.head())

print("Test object-score table:")
display(test_score_df.head())

Validation object-score table:


,object_id,source_image,n_pixels_total,n_pixels_decision,n_pixels_core,n_pixels_border,decision_used_core,border_width,n_predicted_peanut_pixels,peanut_pixel_ratio,peanut_pixel_ratio_smooth,logit_peanut_ratio,stat_norm_mean,stat_norm_median,stat_norm_q10,stat_norm_q90,margin_norm_mean,margin_norm_median,margin_norm_q10,margin_norm_q90,H_mean,Q_mean,true_peanut_pixel_ratio_total,true_peanut_object,area_pixels,batch,sample_kind,object_nut_type
0,almond3_obj001,almond3,63,42,42,21,True,1,11,0.261905,0.267442,-1.007641,1.765927,1.782935,0.604116,2.799349,-0.765927,-0.782935,-1.799349,0.395884,8.513329,3.391019e-07,0.0,False,63,3,pure,almond
1,almond3_obj002,almond3,108,83,83,25,True,1,39,0.469880,0.470238,-0.119189,1.043057,1.046896,0.379714,1.654973,-0.043057,-0.046896,-0.654973,0.620286,6.203417,1.987523e-07,0.0,False,108,3,pure,almond
2,almond3_obj003,almond3,83,61,61,22,True,1,39,0.639344,0.637097,0.562785,0.896054,0.831070,0.414459,1.460637,0.103946,0.168930,-0.460637,0.585541,10.503050,1.639573e-07,0.0,False,83,3,pure,almond
3,almond3_obj004,almond3,90,67,67,23,True,1,45,0.671642,0.669118,0.704197,0.828283,0.730730,0.477525,1.261755,0.171717,0.269270,-0.261755,0.522475,2.449396,1.610749e-07,0.0,False,90,3,pure,almond
4,almond3_obj005,almond3,58,34,34,24,True,1,11,0.323529,0.328571,-0.714653,1.289831,1.357109,0.433901,1.807886,-0.289831,-0.357109,-0.807886,0.566099,4.010342,2.505744e-07,0.0,False,58,3,pure,almond


Test object-score table:


,object_id,source_image,n_pixels_total,n_pixels_decision,n_pixels_core,n_pixels_border,decision_used_core,border_width,n_predicted_peanut_pixels,peanut_pixel_ratio,peanut_pixel_ratio_smooth,logit_peanut_ratio,stat_norm_mean,stat_norm_median,stat_norm_q10,stat_norm_q90,margin_norm_mean,margin_norm_median,margin_norm_q10,margin_norm_q90,H_mean,Q_mean,true_peanut_pixel_ratio_total,true_peanut_object,area_pixels,batch,sample_kind,object_nut_type
0,almond4_obj001,almond4,107,75,75,32,True,1,37,0.493333,0.493421,-0.026317,1.089545,1.019801,0.484577,1.711997,-0.089545,-0.019801,-0.711997,0.515423,9.780658,2.032826e-07,0.0,False,107,4,pure,almond
1,almond4_obj002,almond4,188,145,145,43,True,1,46,0.317241,0.318493,-0.760705,1.315717,1.253431,0.668008,2.115488,-0.315717,-0.253431,-1.115488,0.331992,23.196936,2.305517e-07,0.0,False,188,4,pure,almond
2,almond4_obj003,almond4,113,80,80,33,True,1,56,0.700000,0.697531,0.835568,0.981639,0.836143,0.565283,1.316265,0.018361,0.163857,-0.316265,0.434717,3.443441,1.901890e-07,0.0,False,113,4,pure,almond
3,almond4_obj004,almond4,139,113,113,26,True,1,89,0.787611,0.785088,1.295566,0.763550,0.724015,0.397745,1.205140,0.236450,0.275985,-0.205140,0.602255,4.936508,1.449743e-07,0.0,False,139,4,pure,almond
4,almond4_obj005,almond4,64,50,50,14,True,1,2,0.040000,0.049020,-2.965273,3.692809,3.542313,1.242882,5.422439,-2.692809,-2.542313,-4.422439,-0.242882,61.668022,6.515956e-07,0.0,False,64,4,pure,almond


In [24]:
fig = px.histogram(
    validation_score_df,
    x="peanut_pixel_ratio",
    color="true_peanut_object",
    nbins=30,
    marginal="box",
    barmode="overlay",
    opacity=0.65,
    title="Validation — peanut_pixel_ratio by true class",
)
fig.show()

fig = px.histogram(
    test_score_df,
    x="peanut_pixel_ratio",
    color="true_peanut_object",
    nbins=30,
    marginal="box",
    barmode="overlay",
    opacity=0.65,
    title="Test — peanut_pixel_ratio by true class",
)
fig.show()

fig = px.scatter(
    validation_score_df,
    x="peanut_pixel_ratio",
    y="stat_norm_median",
    color="true_peanut_object",
    hover_data=["object_id", "source_image", "batch", "n_pixels_decision"],
    title="Validation — object score space",
)
fig.show()

In [25]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


OBJECT_SCORE_FEATURES = [
    "logit_peanut_ratio",
    "stat_norm_median",
    "stat_norm_q90",
    "margin_norm_mean",
    "margin_norm_q10",
    "n_pixels_decision",
]

train_score_df = validation_score_df.dropna(
    subset=["true_peanut_object"]
).copy()

X_score_train = train_score_df[OBJECT_SCORE_FEATURES].to_numpy(dtype=float)
y_score_train = train_score_df["true_peanut_object"].astype(int).to_numpy()

object_score_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        C=1.0,
        class_weight=None,
        random_state=RANDOM_STATE,
    ),
)

object_score_model.fit(X_score_train, y_score_train)

print("Object-score model fitted on validation batch.")
print("Features:", OBJECT_SCORE_FEATURES)

Object-score model fitted on validation batch.
Features: ['logit_peanut_ratio', 'stat_norm_median', 'stat_norm_q90', 'margin_norm_mean', 'margin_norm_q10', 'n_pixels_decision']


In [26]:
def add_object_score_probabilities(df, model, feature_cols):
    out = df.copy()
    X = out[feature_cols].to_numpy(dtype=float)
    out["p_peanut_object"] = model.predict_proba(X)[:, 1]
    return out


validation_score_df = add_object_score_probabilities(
    validation_score_df,
    model=object_score_model,
    feature_cols=OBJECT_SCORE_FEATURES,
)

test_score_df = add_object_score_probabilities(
    test_score_df,
    model=object_score_model,
    feature_cols=OBJECT_SCORE_FEATURES,
)


def safe_auc_report(df, set_name):
    d = df.dropna(subset=["p_peanut_object", "true_peanut_object"]).copy()
    y = d["true_peanut_object"].astype(int).to_numpy()
    p = d["p_peanut_object"].to_numpy(dtype=float)

    if len(np.unique(y)) < 2:
        print(f"{set_name}: only one class available, AUC not defined.")
        return

    print(f"{set_name} ROC-AUC: {roc_auc_score(y, p):.3f}")
    print(f"{set_name} average precision: {average_precision_score(y, p):.3f}")


safe_auc_report(validation_score_df, "Validation")
safe_auc_report(test_score_df, "Test")

fig = px.histogram(
    validation_score_df,
    x="p_peanut_object",
    color="true_peanut_object",
    nbins=30,
    marginal="box",
    barmode="overlay",
    opacity=0.65,
    title="Validation — object-level peanut probability",
)
fig.show()

fig = px.histogram(
    test_score_df,
    x="p_peanut_object",
    color="true_peanut_object",
    nbins=30,
    marginal="box",
    barmode="overlay",
    opacity=0.65,
    title="Test — object-level peanut probability",
)
fig.show()

Validation ROC-AUC: 0.997
Validation average precision: 0.997
Test ROC-AUC: 0.976
Test average precision: 0.961


In [27]:
LAMBDA_FN = 2
LAMBDA_FN = 5
LAMBDA_FN = 10

In [28]:
def probability_threshold_grid(
    df,
    prob_col="p_peanut_object",
    true_col="true_peanut_object",
    thresholds=np.linspace(0.01, 0.99, 99),
    lambda_fn=5.0,
):
    rows = []

    d = df.dropna(subset=[prob_col, true_col]).copy()

    for thr in thresholds:
        tmp = d.copy()
        tmp["predicted_peanut_object"] = tmp[prob_col] >= float(thr)

        metrics = binary_detection_metrics(
            tmp,
            true_col=true_col,
            pred_col="predicted_peanut_object",
        )

        metrics["prob_threshold"] = float(thr)
        metrics["lambda_fn"] = float(lambda_fn)
        metrics["cost"] = (
            float(lambda_fn) * metrics["fn_rate"]
            + metrics["fp_rate"]
        )

        rows.append(metrics)

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["cost", "fn_rate", "fp_rate", "f1_score", "accuracy"],
            ascending=[True, True, True, False, False],
        )
        .reset_index(drop=True)
    )


LAMBDA_FN = 20

prob_grid_validation = probability_threshold_grid(
    validation_score_df,
    lambda_fn=LAMBDA_FN,
)

display(prob_grid_validation.head(20))

SELECTED_PROB_THRESHOLD = float(prob_grid_validation.iloc[0]["prob_threshold"])

print("Selected probability threshold:", SELECTED_PROB_THRESHOLD)
print("Lambda FN:", LAMBDA_FN)

,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,prob_threshold,lambda_fn,cost
0,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.19,20.0,0.090909
1,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.20,20.0,0.090909
2,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.21,20.0,0.090909
3,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.22,20.0,0.090909
4,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.23,20.0,0.090909
5,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.24,20.0,0.090909
6,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.25,20.0,0.090909
7,108,53,0,5,50,1.0,0.909091,0.954545,0.953704,0.913793,0.954955,0.0,0.090909,0.26,20.0,0.090909
8,108,53,0,6,49,1.0,0.890909,0.945455,0.944444,0.898305,0.946429,0.0,0.109091,0.16,20.0,0.109091
9,108,53,0,6,49,1.0,0.890909,0.945455,0.944444,0.898305,0.946429,0.0,0.109091,0.17,20.0,0.109091


Selected probability threshold: 0.19
Lambda FN: 20


In [29]:
def evaluate_probability_object_model(
    df,
    set_name,
    prob_threshold,
    model_name="F_object_score_logistic",
):
    out = df.dropna(subset=["p_peanut_object", "true_peanut_object"]).copy()
    out["predicted_peanut_object"] = out["p_peanut_object"] >= float(prob_threshold)

    metrics = binary_detection_metrics(
        out,
        true_col="true_peanut_object",
        pred_col="predicted_peanut_object",
    )

    metrics.update({
        "model_name": model_name,
        "decision_model": "logistic_object_score",
        "set": set_name,
        "preprocessing": str(object_score_base_row["preprocessing"]),
        "n_components": int(object_score_base_row["n_components"]),
        "rule_variant": str(object_score_base_row["rule_variant"]),
        "border_width": int(object_score_base_row["border_width"]),
        "object_threshold": np.nan,
        "prob_threshold": float(prob_threshold),
        "lambda_fn": float(LAMBDA_FN),
        "selection_logic": "object-score probability model",
    })

    return metrics, out


validation_prob_metrics, validation_prob_object_df = evaluate_probability_object_model(
    validation_score_df,
    set_name="validation_score_model_train",
    prob_threshold=SELECTED_PROB_THRESHOLD,
)

test_prob_metrics, test_prob_object_df = evaluate_probability_object_model(
    test_score_df,
    set_name="test",
    prob_threshold=SELECTED_PROB_THRESHOLD,
)

object_score_comparison_df = add_orientation_columns(
    pd.DataFrame([validation_prob_metrics, test_prob_metrics])
)

combined_model_comparison_df = pd.concat(
    [
        hard_model_comparison_df,
        object_score_comparison_df,
    ],
    axis=0,
    ignore_index=True,
    sort=False,
)

comparison_cols = [
    "model_name",
    "decision_model",
    "set",
    "tp", "fn", "fp", "tn",
    "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity",
    "accuracy", "f1_score", "balanced_accuracy",
    "preprocessing", "n_components", "rule_variant",
    "border_width", "object_threshold", "prob_threshold",
    "selection_logic",
]

display(
    combined_model_comparison_df[comparison_cols]
    .sort_values(["set", "fn_rate", "fp_rate", "f1_score", "accuracy"],
                 ascending=[True, True, True, False, False])
    .reset_index(drop=True)
)

,model_name,decision_model,set,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold,prob_threshold,selection_logic
0,C_accuracy,hard_ratio_threshold,test,29,0,6,42,0.000000,0.125000,1.000000,0.875000,0.922078,0.906250,0.937500,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,NaN,max accuracy / F1
1,B_compromise,hard_ratio_threshold,test,29,0,7,41,0.000000,0.145833,1.000000,0.854167,0.909091,0.892308,0.927083,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,NaN,cost compromise
2,F_object_score_logistic,logistic_object_score,test,29,0,8,40,0.000000,0.166667,1.000000,0.833333,0.896104,0.878788,0.916667,absorbance_sg_d1,7,data_driven_emp_cv,1,NaN,0.19,object-score probability model
3,A_safety_FN0,hard_ratio_threshold,test,29,0,9,39,0.000000,0.187500,1.000000,0.812500,0.883117,0.865672,0.906250,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,NaN,"FN=0, then min FP"
4,D_simple_ablation,hard_ratio_threshold,test,29,0,15,33,0.000000,0.312500,1.000000,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,simple_emp_cv,0,0.75,NaN,simple empirical SIMCA ablation
5,E_dd_ablation,hard_ratio_threshold,test,29,0,15,33,0.000000,0.312500,1.000000,0.687500,0.805195,0.794521,0.843750,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75,NaN,DD empirical SIMCA ablation
6,A_safety_FN0,hard_ratio_threshold,validation,53,0,6,49,0.000000,0.109091,1.000000,0.890909,0.944444,0.946429,0.945455,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,NaN,"FN=0, then min FP"
7,D_simple_ablation,hard_ratio_threshold,validation,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,simple_emp_cv,0,0.75,NaN,simple empirical SIMCA ablation
8,E_dd_ablation,hard_ratio_threshold,validation,53,0,10,45,0.000000,0.181818,1.000000,0.818182,0.907407,0.913793,0.909091,absorbance_sg_d1,6,data_driven_emp_cv,0,0.75,NaN,DD empirical SIMCA ablation
9,B_compromise,hard_ratio_threshold,validation,52,1,4,51,0.018868,0.072727,0.981132,0.927273,0.953704,0.954128,0.954202,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,NaN,cost compromise


In [30]:
FINAL_BASELINE_MODEL_NAME = "B_compromise"  # options: A_safety_FN0, B_compromise, C_accuracy, D_simple_ablation, E_dd_ablation
USE_OBJECT_SCORE_MODEL = True              # set True only if you want to deploy the logistic object-score model

final_baseline_row = (
    candidate_models
    .loc[candidate_models["model_name"].eq(FINAL_BASELINE_MODEL_NAME)]
    .iloc[0]
    .copy()
)

SELECTED_PREPROCESSING = str(final_baseline_row["preprocessing"])
SELECTED_PREPROCESSING_STEPS = tuple(SEARCH_PREPROCESSING_CONFIGS[SELECTED_PREPROCESSING])
SELECTED_N_COMPONENTS = int(final_baseline_row["n_components"])
SELECTED_RULE_VARIANT = str(final_baseline_row["rule_variant"])
SELECTED_ALPHA = FIXED_ALPHA
SELECTED_BORDER_WIDTH = int(final_baseline_row["border_width"])
SELECTED_OBJECT_THRESHOLD = float(final_baseline_row["object_threshold"])

print("Final baseline selected for downstream diagnostics/deployment:")
print("model:", FINAL_BASELINE_MODEL_NAME)
print("preprocessing:", SELECTED_PREPROCESSING, SELECTED_PREPROCESSING_STEPS)
print("n_components:", SELECTED_N_COMPONENTS)
print("rule_variant:", SELECTED_RULE_VARIANT)
print("border_width:", SELECTED_BORDER_WIDTH)
print("object_threshold:", SELECTED_OBJECT_THRESHOLD)
print("use object-score model:", USE_OBJECT_SCORE_MODEL)

Final baseline selected for downstream diagnostics/deployment:
model: B_compromise
preprocessing: absorbance_sg_d1 ('absorbance', 'sg_d1')
n_components: 7
rule_variant: data_driven_emp_cv
border_width: 1
object_threshold: 0.8
use object-score model: True


### Simpler versions

In [31]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


SIMPLE_OBJECT_SCORE_FEATURE_SETS = {
    "ratio_only": [
        "logit_peanut_ratio",
    ],
    "ratio_plus_margin_mean": [
        "logit_peanut_ratio",
        "margin_norm_mean",
    ],
    "ratio_plus_stat_median": [
        "logit_peanut_ratio",
        "stat_norm_median",
    ],
    "compact_margin": [
        "logit_peanut_ratio",
        "margin_norm_mean",
        "margin_norm_q10",
    ],
    "compact_stat_margin": [
        "logit_peanut_ratio",
        "stat_norm_median",
        "stat_norm_q90",
        "margin_norm_mean",
    ],
    "current_full": [
        "logit_peanut_ratio",
        "stat_norm_median",
        "stat_norm_q90",
        "margin_norm_mean",
        "margin_norm_q10",
        "n_pixels_decision",
    ],
}


LOGISTIC_C_VALUES = [1.0, 0.3, 0.1, 0.03, 0.01]
LAMBDA_FN_VALUES = [2.0, 5.0, 10.0]

In [32]:
def fit_simple_object_score_model(
    train_df,
    feature_cols,
    C=1.0,
    class_weight=None,
    random_state=42,
):
    d = train_df.dropna(subset=["true_peanut_object"]).copy()

    X = d[feature_cols].to_numpy(dtype=float)
    y = d["true_peanut_object"].astype(int).to_numpy()

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            C=float(C),
            class_weight=class_weight,
            random_state=random_state,
        ),
    )

    model.fit(X, y)
    return model


def add_probabilities_from_model(df, model, feature_cols, prob_col="p_peanut_object"):
    out = df.copy()
    X = out[feature_cols].to_numpy(dtype=float)
    out[prob_col] = model.predict_proba(X)[:, 1]
    return out


def probability_threshold_grid_local(
    df,
    prob_col="p_peanut_object",
    true_col="true_peanut_object",
    thresholds=np.linspace(0.01, 0.99, 99),
    lambda_fn=5.0,
):
    rows = []

    d = df.dropna(subset=[prob_col, true_col]).copy()

    for thr in thresholds:
        tmp = d.copy()
        tmp["predicted_peanut_object"] = tmp[prob_col] >= float(thr)

        metrics = binary_detection_metrics(
            tmp,
            true_col=true_col,
            pred_col="predicted_peanut_object",
        )

        metrics["prob_threshold"] = float(thr)
        metrics["lambda_fn"] = float(lambda_fn)
        metrics["cost"] = (
            float(lambda_fn) * metrics["fn_rate"]
            + metrics["fp_rate"]
        )

        rows.append(metrics)

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["cost", "fn_rate", "fp_rate", "f1_score", "accuracy"],
            ascending=[True, True, True, False, False],
        )
        .reset_index(drop=True)
    )


def evaluate_simple_object_score_models(
    validation_score_df,
    test_score_df,
    mixture_score_df=None,
    feature_sets=SIMPLE_OBJECT_SCORE_FEATURE_SETS,
    C_values=LOGISTIC_C_VALUES,
    lambda_values=LAMBDA_FN_VALUES,
    random_state=42,
):
    rows = []
    stored = {}

    eval_sets = {
        "validation_train": validation_score_df,
        "test": test_score_df,
    }

    if mixture_score_df is not None:
        eval_sets["mixture"] = mixture_score_df

    for feature_set_name, feature_cols in feature_sets.items():
        for C in C_values:
            model = fit_simple_object_score_model(
                train_df=validation_score_df,
                feature_cols=feature_cols,
                C=C,
                class_weight=None,
                random_state=random_state,
            )

            validation_with_prob = add_probabilities_from_model(
                validation_score_df,
                model=model,
                feature_cols=feature_cols,
            )

            for lambda_fn in lambda_values:
                threshold_df = probability_threshold_grid_local(
                    validation_with_prob,
                    lambda_fn=lambda_fn,
                )

                selected_threshold = float(threshold_df.iloc[0]["prob_threshold"])

                key = (feature_set_name, float(C), float(lambda_fn))
                stored[key] = {
                    "model": model,
                    "features": feature_cols,
                    "selected_threshold": selected_threshold,
                    "validation_threshold_grid": threshold_df,
                }

                for set_name, score_df in eval_sets.items():
                    score_df_prob = add_probabilities_from_model(
                        score_df,
                        model=model,
                        feature_cols=feature_cols,
                    )

                    score_df_prob["predicted_peanut_object"] = (
                        score_df_prob["p_peanut_object"] >= selected_threshold
                    )

                    metrics = binary_detection_metrics(
                        score_df_prob,
                        true_col="true_peanut_object",
                        pred_col="predicted_peanut_object",
                    )

                    metrics.update({
                        "feature_set": feature_set_name,
                        "features": ", ".join(feature_cols),
                        "C": float(C),
                        "lambda_fn": float(lambda_fn),
                        "prob_threshold": selected_threshold,
                        "set": set_name,
                        "decision_model": "simple_logistic_object_score",
                    })

                    rows.append(metrics)

    result_df = pd.DataFrame(rows)

    return result_df, stored

## 10. Error diagnostics

Use these plots to understand whether FP/FN are mostly on borders or in the core of the object.

In [33]:
print("Validation pixel errors by border zone")
display(summarize_pixel_errors_by_border_zone(
    pixel_df=validation_selected["pixel_df"],
    object_db=object_db,
    border_width=SELECTED_BORDER_WIDTH,
))

print("Test pixel errors by border zone")
display(summarize_pixel_errors_by_border_zone(
    pixel_df=test_selected["pixel_df"],
    object_db=object_db,
    border_width=SELECTED_BORDER_WIDTH,
))

Validation pixel errors by border zone


,zone,border_width,n_pixels,tp,tn,fp,fn,fp_rate,fn_rate,pixel_accuracy
0,border,1,2026,872,518,506,130,0.494141,0.129741,0.686081
1,core,1,4786,2143,1419,1172,52,0.452335,0.023690,0.744254


Test pixel errors by border zone


,zone,border_width,n_pixels,tp,tn,fp,fn,fp_rate,fn_rate,pixel_accuracy
0,border,1,2084,704,564,717,99,0.559719,0.123288,0.608445
1,core,1,6317,2309,1973,1983,52,0.501264,0.022025,0.677853


In [34]:
def add_object_error_type(object_df):
    df = object_df.copy()
    df["object_error_type"] = "NA"
    mask = df["true_peanut_object"].notna() & df["predicted_peanut_object"].notna()
    y_true = df.loc[mask, "true_peanut_object"].astype(bool)
    y_pred = df.loc[mask, "predicted_peanut_object"].astype(bool)
    df.loc[mask & y_true & y_pred, "object_error_type"] = "TP"
    df.loc[mask & (~y_true) & (~y_pred), "object_error_type"] = "TN"
    df.loc[mask & (~y_true) & y_pred, "object_error_type"] = "FP"
    df.loc[mask & y_true & (~y_pred), "object_error_type"] = "FN"
    return df

validation_object_df = add_object_error_type(validation_object_df)
test_object_df = add_object_error_type(test_object_df)

print("Validation object errors")
display(validation_object_df.groupby(["source_image", "object_error_type"], as_index=False).size())
print("Test object errors")
display(test_object_df.groupby(["source_image", "object_error_type"], as_index=False).size())

Validation object errors


,source_image,object_error_type,size
0,almond3,FP,10
1,almond3,TN,45
2,peanut3,TP,53


Test object errors


,source_image,object_error_type,size
0,almond4,FP,15
1,almond4,TN,33
2,peanut4,TP,29


In [35]:
# Visualize the most problematic test images.
error_images = (
    test_object_df[test_object_df["object_error_type"].isin(["FP", "FN"])]
    ["source_image"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

for image_key in error_images[:6]:
    plot_pixel_error_overlay(
        image_key=image_key,
        image_db=image_db,
        pixel_df=test_selected["pixel_df"],
        title=f"Test pixel error map — {image_key}",
    )

## 11. Optional deployment projection on mixtures

This section is separate from model selection and test evaluation. Once the orientation is reported, you may refit on all pure peanut batches before projecting mixtures. Keep the selected preprocessing, number of components, rule and object decision parameters fixed.

In [36]:
TRAIN_FILTERS_DEPLOYMENT = {
    "sample_kind": ["pure"],
    "object_nut_type": ["peanut"],
    "batch": [1, 2, 3, 4],
}

# Recalibrate empirical 95% thresholds on all available pure peanut batches.
deployment_cv_df, deployment_cv_thresholds = calibrate_simca_thresholds_cv(
    object_db=object_db,
    train_filters=TRAIN_FILTERS_DEPLOYMENT,
    matrix_method=FIXED_MATRIX_METHOD,
    preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
    n_components=SELECTED_N_COMPONENTS,
    alpha=SELECTED_ALPHA,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
    group_col="object_id",
    n_splits=CV_N_SPLITS,
)

deployment_bundle = fit_final_simca_model(
    object_db=object_db,
    train_filters=TRAIN_FILTERS_DEPLOYMENT,
    matrix_method=FIXED_MATRIX_METHOD,
    preprocessing_steps=SELECTED_PREPROCESSING_STEPS,
    n_components=SELECTED_N_COMPONENTS,
    alpha=SELECTED_ALPHA,
    m=FIXED_M,
    random_state=RANDOM_STATE,
    replace=False,
    wavelengths=wavelengths,
    sg_window_length=FIXED_SG_WINDOW_LENGTH,
    sg_polyorder=FIXED_SG_POLYORDER,
)

mixture_pixel_wide_df, _, _ = project_pixels_with_rule_variants(
    object_db=object_db,
    final_bundle=deployment_bundle,
    projection_filters=PROJECTION_FILTERS_MIXTURES,
    cv_thresholds=deployment_cv_thresholds,
    rule_variants=[SELECTED_RULE_VARIANT],
)

mixture_pixel_wide_df = add_pixel_truth_labels(
    pixel_df=mixture_pixel_wide_df,
    image_db=image_db,
    object_db=object_db,
    dilation_radius=FIXED_POSITION_DILATION_RADIUS,
)

mixture_pixel_df = mixture_pixel_wide_df.copy()
mixture_pixel_df["predicted_peanut_pixel"] = mixture_pixel_df[f"pred_{SELECTED_RULE_VARIANT}"].astype(bool)
mixture_pixel_df["rule_statistic"] = mixture_pixel_df[f"stat_{SELECTED_RULE_VARIANT}"]
mixture_pixel_df["rule_limit"] = mixture_pixel_df[f"limit_{SELECTED_RULE_VARIANT}"]

mixture_object_df = aggregate_pixel_predictions_to_objects_core(
    pixel_df=mixture_pixel_df,
    object_db=object_db,
    object_threshold=SELECTED_OBJECT_THRESHOLD,
    border_width=SELECTED_BORDER_WIDTH,
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

display(mixture_object_df.head())

c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


,object_id,source_image,n_pixels_total,n_pixels_core,n_pixels_border,n_pixels_decision,decision_used_core,border_width,min_core_pixels,n_predicted_peanut_pixels,peanut_pixel_ratio,predicted_peanut_object,predicted_label_object,object_threshold,true_peanut_pixel_ratio_decision,true_peanut_pixel_ratio_total,true_peanut_object,true_label_object,area_pixels,batch,sample_kind,object_nut_type,centroid_row,centroid_col
0,alm1pea1_obj001,alm1pea1,84,56,28,56,True,1,20,25,0.446429,False,non_peanut,0.8,0.0,0.0,False,non_peanut,84,None,mixture,unknown,85.726190,45.107143
1,alm1pea1_obj002,alm1pea1,73,56,17,56,True,1,20,12,0.214286,False,non_peanut,0.8,0.0,0.0,False,non_peanut,73,None,mixture,unknown,87.739726,123.767123
2,alm1pea1_obj003,alm1pea1,91,68,23,68,True,1,20,68,1.000000,True,peanut,0.8,1.0,1.0,True,peanut,91,None,mixture,unknown,90.032967,157.340659
3,alm1pea1_obj004,alm1pea1,73,55,18,55,True,1,20,55,1.000000,True,peanut,0.8,1.0,1.0,True,peanut,73,None,mixture,unknown,93.589041,90.164384
4,alm1pea1_obj005,alm1pea1,126,99,27,99,True,1,20,65,0.656566,False,non_peanut,0.8,0.0,0.0,False,non_peanut,126,None,mixture,unknown,98.825397,70.579365


In [37]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==False) & (mixture_object_df['true_peanut_object']==True)].shape[0]

2

In [38]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==True) & (mixture_object_df['true_peanut_object']==False)].shape[0]

74

In [39]:
mixture_object_df[(mixture_object_df['predicted_peanut_object']==True) & (mixture_object_df['true_peanut_object']==False)]['source_image'].value_counts()

source_image
alm4pea3    13
alm4pea2    11
alm4pea4    10
alm3pea1     9
alm3pea4     9
alm4pea1     8
alm3pea2     5
alm3pea3     5
alm1pea4     4
Name: count, dtype: int64

## 11bis. Optional object-score prediction on mixtures

This section applies the logistic object-score model to the mixture projection set.

This is exploratory: mixtures are not used to select the model.

In [40]:
if USE_OBJECT_SCORE_MODEL:
    # Project mixtures with the same base spectral model used by the object-score model.
    mixture_score_projection = get_candidate_projection(
        candidate_row=object_score_base_row,
        projection_filters=PROJECTION_FILTERS_MIXTURES,
        projection_name="mixture_object_score_deployment",
        train_filters=TRAIN_FILTERS_DEPLOYMENT,
    )

    mixture_score_df = aggregate_object_score_features(
        pixel_df=mixture_score_projection["pixel_df"],
        object_db=object_db,
        border_width=int(object_score_base_row["border_width"]),
        min_core_pixels=MIN_CORE_PIXELS,
        fallback_to_all_pixels=True,
    )

    mixture_score_df = add_object_score_probabilities(
        mixture_score_df,
        model=object_score_model,
        feature_cols=OBJECT_SCORE_FEATURES,
    )

    mixture_score_df["predicted_peanut_object"] = (
        mixture_score_df["p_peanut_object"] >= SELECTED_PROB_THRESHOLD
    )

    mixture_score_metrics = binary_detection_metrics(
        mixture_score_df,
        true_col="true_peanut_object",
        pred_col="predicted_peanut_object",
    )

    print("Mixture object-score metrics:")
    display(pd.DataFrame([mixture_score_metrics]))

    display(
        mixture_score_df[
            [
                "object_id",
                "source_image",
                "true_peanut_object",
                "predicted_peanut_object",
                "p_peanut_object",
                "peanut_pixel_ratio",
                "stat_norm_median",
                "margin_norm_mean",
                "n_pixels_decision",
            ]
        ].sort_values("p_peanut_object", ascending=False)
    )

else:
    print("USE_OBJECT_SCORE_MODEL is False. Keeping hard object-threshold predictions for mixtures.")

c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Mixture object-score metrics:


,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate
0,722,145,1,85,491,0.993151,0.852431,0.922791,0.880886,0.630435,0.771277,0.006849,0.147569


,object_id,source_image,true_peanut_object,predicted_peanut_object,p_peanut_object,peanut_pixel_ratio,stat_norm_median,margin_norm_mean,n_pixels_decision
120,alm1pea3_obj039,alm1pea3,True,True,9.960712e-01,1.000000,0.201902,0.797369,26
86,alm1pea3_obj005,alm1pea3,True,True,9.958464e-01,1.000000,0.187352,0.806022,34
9,alm1pea1_obj010,alm1pea1,True,True,9.955536e-01,1.000000,0.208684,0.768519,23
156,alm2pea1_obj008,alm2pea1,True,True,9.953796e-01,1.000000,0.185399,0.790504,39
173,alm2pea1_obj025,alm2pea1,True,True,9.953433e-01,1.000000,0.188584,0.799886,39
...,...,...,...,...,...,...,...,...,...
675,alm5pea3_obj017,alm5pea3,False,False,3.828045e-14,0.027778,5.338985,-4.241246,72
689,alm5pea3_obj031,alm5pea3,False,False,5.127858e-15,0.000000,5.215539,-4.669238,63
692,alm5pea3_obj034,alm5pea3,False,False,1.313243e-15,0.020690,5.915698,-4.580999,145
285,alm2pea4_obj013,alm2pea4,False,False,2.474467e-16,0.000000,5.696245,-5.044831,106


In [41]:
mixture_score_df[(mixture_score_df['predicted_peanut_object']==False) & (mixture_score_df['true_peanut_object']==True)].shape[0]

1

In [42]:
mixture_score_df[(mixture_score_df['predicted_peanut_object']==True) & (mixture_score_df['true_peanut_object']==False)].shape[0]

85

In [43]:
mixture_score_df[(mixture_score_df['predicted_peanut_object']==True) & (mixture_score_df['true_peanut_object']==False)]['source_image'].value_counts()

source_image
alm4pea2    15
alm4pea3    14
alm3pea1    12
alm3pea2    10
alm4pea1    10
alm3pea4     7
alm4pea4     7
alm3pea3     6
alm1pea4     3
alm1pea3     1
Name: count, dtype: int64

## 12. Final clean model comparison

This section compares the final selected baseline models in a controlled way.

For validation and test:
- the model is fitted only on calibration batches 1–2.

For mixtures:
- the model is refitted on all pure peanut batches 1–4;
- the hyperparameters are not changed;
- mixtures remain a projection/application set.

In [44]:
FINAL_CANDIDATE_MODELS = pd.DataFrame([
    {
        "model_name": "A_safety_FN0",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.75,
        "selection_logic": "FN=0 orientation, then min FP",
    },
    {
        "model_name": "B_compromise",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.80,
        "selection_logic": "FN/FP compromise",
    },
    {
        "model_name": "C_accuracy",
        "preprocessing": "absorbance_sg_d1",
        "n_components": 7,
        "rule_variant": "data_driven_emp_cv",
        "border_width": 1,
        "object_threshold": 0.85,
        "selection_logic": "max accuracy / F1 orientation",
    },
])

display(FINAL_CANDIDATE_MODELS)

,model_name,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
0,A_safety_FN0,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,"FN=0 orientation, then min FP"
1,B_compromise,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,FN/FP compromise
2,C_accuracy,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1 orientation


In [45]:
final_projection_cache = {}


def _filters_to_key(filters):
    items = []

    for key, value in sorted(filters.items()):
        if isinstance(value, (list, tuple, set, np.ndarray)):
            value_key = tuple(value)
        else:
            value_key = value

        items.append((key, value_key))

    return tuple(items)


def project_candidate_empirical_cv(
    candidate_row,
    projection_filters,
    projection_name,
    train_filters,
    train_role,
    random_state=RANDOM_STATE,
    dilation_radius=FIXED_POSITION_DILATION_RADIUS,
):
    """
    Fit one candidate model, project pixels, and attach pixel-level truth.

    The empirical SIMCA thresholds are recalibrated on the training set with
    alpha = 0.05, i.e. 95% empirical CV quantile.
    """
    preprocessing = str(candidate_row["preprocessing"])
    preprocessing_steps = tuple(SEARCH_PREPROCESSING_CONFIGS[preprocessing])
    n_components = int(candidate_row["n_components"])
    rule_variant = str(candidate_row["rule_variant"])

    cache_key = (
        str(candidate_row["model_name"]),
        _filters_to_key(train_filters),
        _filters_to_key(projection_filters),
        str(projection_name),
        str(train_role),
        preprocessing,
        preprocessing_steps,
        n_components,
        rule_variant,
        float(FIXED_ALPHA),
        int(FIXED_M),
        int(FIXED_SG_WINDOW_LENGTH),
        int(FIXED_SG_POLYORDER),
        int(random_state),
        int(dilation_radius),
    )

    if cache_key in final_projection_cache:
        return final_projection_cache[cache_key]

    cv_df, cv_thresholds = calibrate_simca_thresholds_cv(
        object_db=object_db,
        train_filters=train_filters,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=preprocessing_steps,
        n_components=n_components,
        alpha=FIXED_ALPHA,
        m=FIXED_M,
        random_state=int(random_state),
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
        group_col="object_id",
        n_splits=CV_N_SPLITS,
    )

    final_bundle = fit_final_simca_model(
        object_db=object_db,
        train_filters=train_filters,
        matrix_method=FIXED_MATRIX_METHOD,
        preprocessing_steps=preprocessing_steps,
        n_components=n_components,
        alpha=FIXED_ALPHA,
        m=FIXED_M,
        random_state=int(random_state),
        replace=False,
        wavelengths=wavelengths,
        sg_window_length=FIXED_SG_WINDOW_LENGTH,
        sg_polyorder=FIXED_SG_POLYORDER,
    )

    pixel_wide_df, simca_values, X_pixel = project_pixels_with_rule_variants(
        object_db=object_db,
        final_bundle=final_bundle,
        projection_filters=projection_filters,
        cv_thresholds=cv_thresholds,
        rule_variants=[rule_variant],
    )

    pixel_wide_df = add_pixel_truth_labels(
        pixel_df=pixel_wide_df,
        image_db=image_db,
        object_db=object_db,
        dilation_radius=int(dilation_radius),
    )

    pixel_df = pixel_wide_df.copy()
    pixel_df["predicted_peanut_pixel"] = pixel_df[f"pred_{rule_variant}"].astype(bool)
    pixel_df["predicted_label_pixel"] = np.where(
        pixel_df["predicted_peanut_pixel"],
        "peanut",
        "non_peanut",
    )
    pixel_df["rule_statistic"] = pixel_df[f"stat_{rule_variant}"]
    pixel_df["rule_limit"] = pixel_df[f"limit_{rule_variant}"]
    pixel_df["rule_variant"] = rule_variant
    pixel_df["model_name"] = str(candidate_row["model_name"])
    pixel_df["projection_name"] = str(projection_name)
    pixel_df["train_role"] = str(train_role)
    pixel_df["random_state"] = int(random_state)
    pixel_df["truth_dilation_radius"] = int(dilation_radius)

    out = {
        "candidate_row": candidate_row.copy(),
        "projection_name": projection_name,
        "train_role": train_role,
        "cv_df": cv_df,
        "cv_thresholds": cv_thresholds,
        "final_bundle": final_bundle,
        "pixel_df": pixel_df,
        "simca_values": simca_values,
        "X_pixel": X_pixel,
    }

    final_projection_cache[cache_key] = out
    return out


def evaluate_hard_candidate(
    candidate_row,
    projection_filters,
    projection_name,
    train_filters,
    train_role,
    random_state=RANDOM_STATE,
    dilation_radius=FIXED_POSITION_DILATION_RADIUS,
):
    projection = project_candidate_empirical_cv(
        candidate_row=candidate_row,
        projection_filters=projection_filters,
        projection_name=projection_name,
        train_filters=train_filters,
        train_role=train_role,
        random_state=random_state,
        dilation_radius=dilation_radius,
    )

    object_df = aggregate_pixel_predictions_to_objects_core(
        pixel_df=projection["pixel_df"],
        object_db=object_db,
        object_threshold=float(candidate_row["object_threshold"]),
        border_width=int(candidate_row["border_width"]),
        min_core_pixels=MIN_CORE_PIXELS,
        fallback_to_all_pixels=True,
    )

    metrics = binary_detection_metrics(
        object_df,
        true_col="true_peanut_object",
        pred_col="predicted_peanut_object",
    )

    metrics.update({
        "model_name": str(candidate_row["model_name"]),
        "decision_model": "hard_ratio_threshold",
        "set": str(projection_name),
        "train_role": str(train_role),
        "preprocessing": str(candidate_row["preprocessing"]),
        "n_components": int(candidate_row["n_components"]),
        "rule_variant": str(candidate_row["rule_variant"]),
        "border_width": int(candidate_row["border_width"]),
        "object_threshold": float(candidate_row["object_threshold"]),
        "random_state": int(random_state),
        "truth_dilation_radius": int(dilation_radius),
        "selection_logic": str(candidate_row["selection_logic"]),
    })

    return metrics, object_df, projection

In [46]:
TRAIN_FILTERS_DEPLOYMENT = {
    "sample_kind": ["pure"],
    "object_nut_type": ["peanut"],
    "batch": [1, 2, 3, 4],
}

FINAL_COMPARISON_SETS = [
    {
        "set": "validation",
        "projection_filters": VALIDATION_FILTERS,
        "train_filters": TRAIN_FILTERS_CALIBRATION,
        "train_role": "calibration_batches_1_2",
    },
    {
        "set": "test",
        "projection_filters": TEST_FILTERS,
        "train_filters": TRAIN_FILTERS_CALIBRATION,
        "train_role": "calibration_batches_1_2",
    },
    {
        "set": "mixture",
        "projection_filters": PROJECTION_FILTERS_MIXTURES,
        "train_filters": TRAIN_FILTERS_DEPLOYMENT,
        "train_role": "deployment_pure_batches_1_2_3_4",
    },
]

final_hard_rows = []
final_model_outputs = {}

for _, candidate_row in FINAL_CANDIDATE_MODELS.iterrows():
    model_name = str(candidate_row["model_name"])

    for set_info in FINAL_COMPARISON_SETS:
        metrics, object_df, projection = evaluate_hard_candidate(
            candidate_row=candidate_row,
            projection_filters=set_info["projection_filters"],
            projection_name=set_info["set"],
            train_filters=set_info["train_filters"],
            train_role=set_info["train_role"],
            random_state=RANDOM_STATE,
            dilation_radius=FIXED_POSITION_DILATION_RADIUS,
        )

        final_hard_rows.append(metrics)

        final_model_outputs[(model_name, set_info["set"])] = {
            "metrics": metrics,
            "object_df": object_df,
            "pixel_df": projection["pixel_df"],
            "projection": projection,
        }

final_hard_comparison_df = add_orientation_columns(pd.DataFrame(final_hard_rows))

FINAL_COMPARE_COLS = [
    "model_name",
    "decision_model",
    "set",
    "train_role",
    "tp", "fn", "fp", "tn",
    "fn_rate", "fp_rate",
    "peanut_sensitivity", "almond_specificity",
    "accuracy", "f1_score", "balanced_accuracy",
    "preprocessing", "n_components", "rule_variant",
    "border_width", "object_threshold",
    "truth_dilation_radius",
    "selection_logic",
]

display(
    final_hard_comparison_df[FINAL_COMPARE_COLS]
    .sort_values(
        ["set", "fn", "fp", "f1_score", "accuracy"],
        ascending=[True, True, True, False, False],
    )
    .reset_index(drop=True)
)

c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

,model_name,decision_model,set,train_role,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold,truth_dilation_radius,selection_logic
0,B_compromise,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,144,2,74,502,0.013699,0.128472,0.986301,0.871528,0.894737,0.791209,0.928915,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
1,A_safety_FN0,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,144,2,88,488,0.013699,0.152778,0.986301,0.847222,0.875346,0.761905,0.916762,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
2,C_accuracy,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,141,5,57,519,0.034247,0.098958,0.965753,0.901042,0.914127,0.819767,0.933398,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation
3,C_accuracy,hard_ratio_threshold,test,calibration_batches_1_2,29,0,6,42,0.000000,0.125000,1.000000,0.875000,0.922078,0.906250,0.937500,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation
4,B_compromise,hard_ratio_threshold,test,calibration_batches_1_2,29,0,7,41,0.000000,0.145833,1.000000,0.854167,0.909091,0.892308,0.927083,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
5,A_safety_FN0,hard_ratio_threshold,test,calibration_batches_1_2,29,0,9,39,0.000000,0.187500,1.000000,0.812500,0.883117,0.865672,0.906250,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
6,A_safety_FN0,hard_ratio_threshold,validation,calibration_batches_1_2,53,0,6,49,0.000000,0.109091,1.000000,0.890909,0.944444,0.946429,0.945455,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
7,B_compromise,hard_ratio_threshold,validation,calibration_batches_1_2,52,1,4,51,0.018868,0.072727,0.981132,0.927273,0.953704,0.954128,0.954202,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
8,C_accuracy,hard_ratio_threshold,validation,calibration_batches_1_2,51,2,2,53,0.037736,0.036364,0.962264,0.963636,0.962963,0.962264,0.962950,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation


## 13. Three-way decision: peanut / non-peanut / uncertain

Binary object decisions force every object into peanut or non-peanut.

Here, we introduce an uncertainty region based on the object peanut pixel ratio.

The decision is:

- non-peanut if peanut_pixel_ratio < low_threshold;
- uncertain if low_threshold <= peanut_pixel_ratio < high_threshold;
- peanut if peanut_pixel_ratio >= high_threshold.

The validation batch is used to select the two thresholds. Test and mixture are then evaluated with fixed thresholds.

In [47]:
def add_three_way_decision(
    df,
    score_col="peanut_pixel_ratio",
    low_threshold=0.70,
    high_threshold=0.90,
    output_col="three_way_decision",
):
    if low_threshold >= high_threshold:
        raise ValueError("low_threshold must be < high_threshold.")

    out = df.copy()
    score = out[score_col].astype(float)

    out[output_col] = np.select(
        [
            score < float(low_threshold),
            score >= float(high_threshold),
        ],
        [
            "non_peanut",
            "peanut",
        ],
        default="uncertain",
    )

    return out


def three_way_metrics(
    df,
    decision_col="three_way_decision",
    true_col="true_peanut_object",
):
    d = df.dropna(subset=[decision_col, true_col]).copy()

    y_true = d[true_col].astype(bool)
    is_peanut_pred = d[decision_col].eq("peanut")
    is_non_peanut_pred = d[decision_col].eq("non_peanut")
    is_uncertain = d[decision_col].eq("uncertain")

    n_total = int(len(d))
    n_uncertain = int(is_uncertain.sum())
    n_decided = int((~is_uncertain).sum())

    n_peanut = int(y_true.sum())
    n_almond = int((~y_true).sum())

    missed_peanut = y_true & is_non_peanut_pred
    auto_false_positive = (~y_true) & is_peanut_pred
    uncertain_peanut = y_true & is_uncertain
    uncertain_almond = (~y_true) & is_uncertain

    out = {
        "n_total": n_total,
        "n_decided": n_decided,
        "n_uncertain": n_uncertain,
        "uncertain_rate": n_uncertain / n_total if n_total > 0 else np.nan,

        "n_peanut": n_peanut,
        "n_almond": n_almond,

        "missed_peanut": int(missed_peanut.sum()),
        "auto_false_positive": int(auto_false_positive.sum()),
        "uncertain_peanut": int(uncertain_peanut.sum()),
        "uncertain_almond": int(uncertain_almond.sum()),

        "missed_peanut_rate": (
            missed_peanut.sum() / n_peanut if n_peanut > 0 else np.nan
        ),
        "auto_fp_rate": (
            auto_false_positive.sum() / n_almond if n_almond > 0 else np.nan
        ),
        "peanut_uncertain_rate": (
            uncertain_peanut.sum() / n_peanut if n_peanut > 0 else np.nan
        ),
        "almond_uncertain_rate": (
            uncertain_almond.sum() / n_almond if n_almond > 0 else np.nan
        ),
    }

    decided = d[~is_uncertain].copy()

    if len(decided) > 0:
        decided["predicted_peanut_object"] = decided[decision_col].eq("peanut")

        decided_metrics = binary_detection_metrics(
            decided,
            true_col=true_col,
            pred_col="predicted_peanut_object",
        )

        for key, value in decided_metrics.items():
            out[f"decided_{key}"] = value

    return out


def three_way_threshold_grid(
    df,
    score_col="peanut_pixel_ratio",
    true_col="true_peanut_object",
    low_values=np.arange(0.50, 0.86, 0.05),
    high_values=np.arange(0.65, 0.96, 0.05),
    lambda_fn=10.0,
    review_penalty=0.20,
):
    rows = []

    for low in low_values:
        for high in high_values:
            if low >= high:
                continue

            tmp = add_three_way_decision(
                df,
                score_col=score_col,
                low_threshold=float(low),
                high_threshold=float(high),
            )

            metrics = three_way_metrics(
                tmp,
                decision_col="three_way_decision",
                true_col=true_col,
            )

            cost = (
                float(lambda_fn) * metrics["missed_peanut_rate"]
                + metrics["auto_fp_rate"]
                + float(review_penalty) * metrics["uncertain_rate"]
            )

            metrics.update({
                "score_col": score_col,
                "low_threshold": float(low),
                "high_threshold": float(high),
                "lambda_fn": float(lambda_fn),
                "review_penalty": float(review_penalty),
                "three_way_cost": float(cost),
            })

            rows.append(metrics)

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "three_way_cost",
                "missed_peanut",
                "auto_false_positive",
                "uncertain_rate",
            ],
            ascending=[True, True, True, True],
        )
        .reset_index(drop=True)
    )

In [48]:
THREE_WAY_BASE_MODEL_NAME = "C_accuracy"

three_way_base_validation_df = final_model_outputs[
    (THREE_WAY_BASE_MODEL_NAME, "validation")
]["object_df"].copy()

three_way_grid_validation = three_way_threshold_grid(
    three_way_base_validation_df,
    score_col="peanut_pixel_ratio",
    low_values=np.round(np.arange(0.50, 0.86, 0.05), 2),
    high_values=np.round(np.arange(0.65, 0.96, 0.05), 2),
    lambda_fn=10.0,
    review_penalty=0.20,
)

display(three_way_grid_validation.head(30))

BEST_THREE_WAY_ROW = three_way_grid_validation.iloc[0].copy()
THREE_WAY_LOW = float(BEST_THREE_WAY_ROW["low_threshold"])
THREE_WAY_HIGH = float(BEST_THREE_WAY_ROW["high_threshold"])

print("Selected three-way thresholds on validation:")
print("low_threshold:", THREE_WAY_LOW)
print("high_threshold:", THREE_WAY_HIGH)
display(pd.DataFrame([BEST_THREE_WAY_ROW]))

,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,score_col,low_threshold,high_threshold,lambda_fn,review_penalty,three_way_cost
0,108,99,9,0.083333,53,55,0,1,4,5,0.0,0.018182,0.075472,0.090909,99,49,0,1,49,1.0,0.980000,0.990000,0.989899,0.980000,0.989899,0.0,0.020000,peanut_pixel_ratio,0.70,0.90,10.0,0.2,0.034848
1,108,99,9,0.083333,53,55,0,1,4,5,0.0,0.018182,0.075472,0.090909,99,49,0,1,49,1.0,0.980000,0.990000,0.989899,0.980000,0.989899,0.0,0.020000,peanut_pixel_ratio,0.75,0.90,10.0,0.2,0.034848
2,108,97,11,0.101852,53,55,0,1,4,7,0.0,0.018182,0.075472,0.127273,97,49,0,1,47,1.0,0.979167,0.989583,0.989691,0.980000,0.989899,0.0,0.020833,peanut_pixel_ratio,0.65,0.90,10.0,0.2,0.038552
3,108,86,22,0.203704,53,55,0,0,16,6,0.0,0.000000,0.301887,0.109091,86,37,0,0,49,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.70,0.95,10.0,0.2,0.040741
4,108,86,22,0.203704,53,55,0,0,16,6,0.0,0.000000,0.301887,0.109091,86,37,0,0,49,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.75,0.95,10.0,0.2,0.040741
5,108,94,14,0.129630,53,55,0,1,4,10,0.0,0.018182,0.075472,0.181818,94,49,0,1,44,1.0,0.977778,0.988889,0.989362,0.980000,0.989899,0.0,0.022222,peanut_pixel_ratio,0.60,0.90,10.0,0.2,0.044108
6,108,84,24,0.222222,53,55,0,0,16,8,0.0,0.000000,0.301887,0.145455,84,37,0,0,47,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.65,0.95,10.0,0.2,0.044444
7,108,93,15,0.138889,53,55,0,1,4,11,0.0,0.018182,0.075472,0.200000,93,49,0,1,43,1.0,0.977273,0.988636,0.989247,0.980000,0.989899,0.0,0.022727,peanut_pixel_ratio,0.55,0.90,10.0,0.2,0.045960
8,108,102,6,0.055556,53,55,0,2,2,4,0.0,0.036364,0.037736,0.072727,102,51,0,2,49,1.0,0.960784,0.980392,0.980392,0.962264,0.980769,0.0,0.039216,peanut_pixel_ratio,0.70,0.85,10.0,0.2,0.047475
9,108,102,6,0.055556,53,55,0,2,2,4,0.0,0.036364,0.037736,0.072727,102,51,0,2,49,1.0,0.960784,0.980392,0.980392,0.962264,0.980769,0.0,0.039216,peanut_pixel_ratio,0.75,0.85,10.0,0.2,0.047475


Selected three-way thresholds on validation:
low_threshold: 0.7
high_threshold: 0.9


,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,score_col,low_threshold,high_threshold,lambda_fn,review_penalty,three_way_cost
0,108,99,9,0.083333,53,55,0,1,4,5,0.0,0.018182,0.075472,0.090909,99,49,0,1,49,1.0,0.98,0.99,0.989899,0.98,0.989899,0.0,0.02,peanut_pixel_ratio,0.7,0.9,10.0,0.2,0.034848


In [49]:
def evaluate_three_way_on_sets(
    object_tables,
    score_col,
    low_threshold,
    high_threshold,
    model_name="three_way_ratio",
):
    rows = []
    tables = {}

    for set_name, df in object_tables.items():
        tmp = add_three_way_decision(
            df,
            score_col=score_col,
            low_threshold=float(low_threshold),
            high_threshold=float(high_threshold),
        )

        metrics = three_way_metrics(tmp)

        metrics.update({
            "model_name": model_name,
            "decision_model": "three_way_uncertain",
            "set": set_name,
            "score_col": score_col,
            "low_threshold": float(low_threshold),
            "high_threshold": float(high_threshold),
            "base_model": THREE_WAY_BASE_MODEL_NAME,
        })

        rows.append(metrics)
        tables[set_name] = tmp

    return pd.DataFrame(rows), tables


three_way_object_tables = {
    "validation": final_model_outputs[(THREE_WAY_BASE_MODEL_NAME, "validation")]["object_df"],
    "test": final_model_outputs[(THREE_WAY_BASE_MODEL_NAME, "test")]["object_df"],
    "mixture": final_model_outputs[(THREE_WAY_BASE_MODEL_NAME, "mixture")]["object_df"],
}

three_way_results_df, three_way_tables = evaluate_three_way_on_sets(
    object_tables=three_way_object_tables,
    score_col="peanut_pixel_ratio",
    low_threshold=THREE_WAY_LOW,
    high_threshold=THREE_WAY_HIGH,
    model_name="three_way_ratio",
)

display(three_way_results_df)

,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,model_name,decision_model,set,score_col,low_threshold,high_threshold,base_model
0,108,99,9,0.083333,53,55,0,1,4,5,0.000000,0.018182,0.075472,0.090909,99,49,0,1,49,1.000000,0.980000,0.990000,0.989899,0.980000,0.989899,0.000000,0.020000,three_way_ratio,three_way_uncertain,validation,peanut_pixel_ratio,0.7,0.9,C_accuracy
1,77,70,7,0.090909,29,48,0,5,1,6,0.000000,0.104167,0.034483,0.125000,70,28,0,5,37,1.000000,0.880952,0.940476,0.928571,0.848485,0.918033,0.000000,0.119048,three_way_ratio,three_way_uncertain,test,peanut_pixel_ratio,0.7,0.9,C_accuracy
2,722,647,75,0.103878,146,576,1,34,9,66,0.006849,0.059028,0.061644,0.114583,647,136,1,34,476,0.992701,0.933333,0.963017,0.945904,0.800000,0.885993,0.007299,0.066667,three_way_ratio,three_way_uncertain,mixture,peanut_pixel_ratio,0.7,0.9,C_accuracy


## 14. Analysis of uncertain objects

This section checks whether uncertain objects are true ambiguous cases, segmentation/truth-border cases, or systematic cases linked to specific mixture images.

The goal is not yet to model the error, but to define which errors are reliable enough to be modelled.

In [50]:
from src.border_decision import add_border_flags_to_pixel_df


def build_object_diagnostic_features(
    pixel_df,
    object_db,
    border_width=1,
    min_core_pixels=20,
    fallback_to_all_pixels=True,
):
    df = pixel_df.copy()

    df = add_border_flags_to_pixel_df(
        pixel_df=df,
        object_db=object_db,
        border_width=int(border_width),
        object_id_col="object_id",
        row_col="row",
        col_col="col",
    )

    df["predicted_peanut_pixel"] = df["predicted_peanut_pixel"].astype(bool)
    df["stat_norm"] = df["rule_statistic"].astype(float) / df["rule_limit"].astype(float)
    df["margin_norm"] = 1.0 - df["stat_norm"]

    rows = []

    for (object_id, source_image), group in df.groupby(["object_id", "source_image"], sort=False):
        core = group[group["is_core_pixel"]].copy()
        decision_group = core
        used_core = True

        if len(core) < int(min_core_pixels):
            if fallback_to_all_pixels:
                decision_group = group.copy()
                used_core = False
            else:
                decision_group = core.copy()

        if len(decision_group) == 0:
            continue

        obj = object_db.get(str(object_id), {})
        centroid = obj.get("centroid", (np.nan, np.nan))

        row = {
            "object_id": object_id,
            "source_image": source_image,
            "n_pixels_total": int(len(group)),
            "n_pixels_decision": int(len(decision_group)),
            "n_pixels_core": int(group["is_core_pixel"].sum()),
            "n_pixels_border": int(group["is_border_pixel"].sum()),
            "decision_used_core": bool(used_core),

            "peanut_pixel_ratio_diag": float(decision_group["predicted_peanut_pixel"].mean()),
            "stat_norm_mean": float(decision_group["stat_norm"].mean()),
            "stat_norm_median": float(decision_group["stat_norm"].median()),
            "stat_norm_q10": float(decision_group["stat_norm"].quantile(0.10)),
            "stat_norm_q90": float(decision_group["stat_norm"].quantile(0.90)),
            "margin_norm_mean": float(decision_group["margin_norm"].mean()),
            "margin_norm_median": float(decision_group["margin_norm"].median()),
            "margin_norm_q10": float(decision_group["margin_norm"].quantile(0.10)),
            "margin_norm_q90": float(decision_group["margin_norm"].quantile(0.90)),
            "H_mean": float(decision_group["H"].mean()) if "H" in decision_group.columns else np.nan,
            "Q_mean": float(decision_group["Q"].mean()) if "Q" in decision_group.columns else np.nan,

            "area_pixels": obj.get("area_pixels", np.nan),
            "batch": obj.get("batch", None),
            "sample_kind": obj.get("sample_kind", None),
            "object_nut_type": obj.get("object_nut_type", None),
            "centroid_row": centroid[0] if len(centroid) > 0 else np.nan,
            "centroid_col": centroid[1] if len(centroid) > 1 else np.nan,
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [51]:
mixture_three_way_df = three_way_tables["mixture"].copy()

mixture_diag_features_df = build_object_diagnostic_features(
    pixel_df=final_model_outputs[(THREE_WAY_BASE_MODEL_NAME, "mixture")]["pixel_df"],
    object_db=object_db,
    border_width=int(
        FINAL_CANDIDATE_MODELS
        .loc[FINAL_CANDIDATE_MODELS["model_name"].eq(THREE_WAY_BASE_MODEL_NAME), "border_width"]
        .iloc[0]
    ),
    min_core_pixels=MIN_CORE_PIXELS,
    fallback_to_all_pixels=True,
)

mixture_three_way_diag_df = mixture_three_way_df.merge(
    mixture_diag_features_df,
    on=["object_id", "source_image"],
    how="left",
    suffixes=("", "_diag"),
)

uncertain_mixture_df = (
    mixture_three_way_diag_df
    .query("three_way_decision == 'uncertain'")
    .copy()
)

print("Uncertain objects by true class:")
display(
    uncertain_mixture_df
    .groupby("true_peanut_object", dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
)

print("Uncertain objects by mixture image:")
display(
    uncertain_mixture_df
    .groupby(["source_image", "true_peanut_object"], dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
    .sort_values(["source_image", "true_peanut_object"])
)

display(
    uncertain_mixture_df[
        [
            "object_id", "source_image",
            "true_peanut_object", "three_way_decision",
            "peanut_pixel_ratio",
            "stat_norm_median", "stat_norm_q90",
            "margin_norm_mean", "margin_norm_q10",
            "H_mean", "Q_mean",
            "n_pixels_decision", "area_pixels",
            "centroid_row", "centroid_col",
        ]
    ]
    .sort_values("peanut_pixel_ratio")
    .reset_index(drop=True)
)

Uncertain objects by true class:


,true_peanut_object,n_objects
0,False,66
1,True,9


Uncertain objects by mixture image:


,source_image,true_peanut_object,n_objects
0,alm1pea1,True,2
1,alm1pea2,True,2
2,alm1pea3,True,1
3,alm1pea4,False,5
4,alm2pea1,True,1
5,alm2pea3,True,1
6,alm3pea1,False,6
7,alm3pea1,True,1
8,alm3pea2,False,7
9,alm3pea3,False,3


,object_id,source_image,true_peanut_object,three_way_decision,peanut_pixel_ratio,stat_norm_median,stat_norm_q90,margin_norm_mean,margin_norm_q10,H_mean,Q_mean,n_pixels_decision,area_pixels,centroid_row,centroid_col
0,alm4pea4_obj011,alm4pea4,False,uncertain,0.700000,0.814300,1.228780,0.172870,-0.228780,6.019162,1.607533e-07,70,98,152.367347,83.469388
1,alm3pea1_obj034,alm3pea1,False,uncertain,0.720930,0.829098,1.418128,0.098197,-0.418128,2.035549,1.814967e-07,43,58,248.431034,96.206897
2,alm4pea4_obj026,alm4pea4,False,uncertain,0.725806,0.842187,1.271609,0.144768,-0.271609,8.728181,1.627681e-07,62,89,243.640449,229.101124
3,alm1pea2_obj037,alm1pea2,True,uncertain,0.727273,0.524374,1.215298,0.113913,-0.215298,40.127746,1.258591e-07,33,43,288.953488,246.627907
4,alm4pea1_obj013,alm4pea1,False,uncertain,0.727273,0.848745,1.180108,0.166339,-0.180108,4.274093,1.644897e-07,55,79,174.860759,64.392405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,alm4pea4_obj030,alm4pea4,False,uncertain,0.887931,0.757131,1.002638,0.238293,-0.002638,20.103256,1.279994e-07,116,156,270.083333,82.769231
71,alm4pea4_obj003,alm4pea4,False,uncertain,0.891156,0.656659,1.001618,0.309661,-0.001618,13.831801,1.220455e-07,147,183,104.830601,70.803279
72,alm3pea2_obj010,alm3pea2,False,uncertain,0.891892,0.612103,0.999291,0.347638,0.000709,4.006629,1.278065e-07,37,62,132.354839,109.403226
73,alm4pea1_obj020,alm4pea1,False,uncertain,0.894737,0.611468,1.007068,0.339865,-0.007068,2.462484,1.315202e-07,57,73,227.561644,129.821918


In [52]:
fig = px.histogram(
    mixture_three_way_diag_df,
    x="peanut_pixel_ratio",
    color="three_way_decision",
    facet_col="true_peanut_object",
    nbins=40,
    marginal="box",
    title="Mixtures — peanut pixel ratio by true class and three-way decision",
)
fig.add_vline(x=THREE_WAY_LOW, line_dash="dash")
fig.add_vline(x=THREE_WAY_HIGH, line_dash="dash")
fig.show()

fig = px.scatter(
    mixture_three_way_diag_df,
    x="peanut_pixel_ratio",
    y="stat_norm_median",
    color="three_way_decision",
    symbol="true_peanut_object",
    hover_data=[
        "object_id",
        "source_image",
        "margin_norm_mean",
        "n_pixels_decision",
        "area_pixels",
    ],
    title="Mixtures — uncertain objects in object-score space",
)
fig.add_vline(x=THREE_WAY_LOW, line_dash="dash")
fig.add_vline(x=THREE_WAY_HIGH, line_dash="dash")
fig.show()

## 15. Per-mixture-image analysis

This section checks whether FP, FN and uncertain objects are uniformly distributed, or concentrated in specific mixture images.

A concentration in a few images suggests a batch/acquisition/position-reference issue rather than a purely spectral SIMCA issue.

In [53]:
def summarize_three_way_by_image(df):
    rows = []

    for source_image, group in df.groupby("source_image", dropna=False):
        y_true = group["true_peanut_object"].astype(bool)

        is_peanut_pred = group["three_way_decision"].eq("peanut")
        is_non_peanut_pred = group["three_way_decision"].eq("non_peanut")
        is_uncertain = group["three_way_decision"].eq("uncertain")

        n_total = int(len(group))
        n_peanut = int(y_true.sum())
        n_almond = int((~y_true).sum())

        row = {
            "source_image": source_image,
            "n_total": n_total,
            "n_peanut": n_peanut,
            "n_almond": n_almond,

            "missed_peanut": int((y_true & is_non_peanut_pred).sum()),
            "auto_false_positive": int(((~y_true) & is_peanut_pred).sum()),
            "uncertain_peanut": int((y_true & is_uncertain).sum()),
            "uncertain_almond": int(((~y_true) & is_uncertain).sum()),

            "uncertain_total": int(is_uncertain.sum()),
            "uncertain_rate": float(is_uncertain.mean()),

            "mean_peanut_pixel_ratio": float(group["peanut_pixel_ratio"].mean()),
            "median_peanut_pixel_ratio": float(group["peanut_pixel_ratio"].median()),
        }

        row["missed_peanut_rate"] = (
            row["missed_peanut"] / n_peanut if n_peanut > 0 else np.nan
        )
        row["auto_fp_rate"] = (
            row["auto_false_positive"] / n_almond if n_almond > 0 else np.nan
        )

        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["missed_peanut", "auto_false_positive", "uncertain_total"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )


mixture_image_summary_df = summarize_three_way_by_image(mixture_three_way_diag_df)

display(mixture_image_summary_df)

,source_image,n_total,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,uncertain_total,uncertain_rate,mean_peanut_pixel_ratio,median_peanut_pixel_ratio,missed_peanut_rate,auto_fp_rate
0,alm2pea2,40,15,25,1,0,0,0,0,0.000000,0.465226,0.338542,0.066667,0.000000
1,alm4pea2,31,1,30,0,8,0,8,8,0.258065,0.686091,0.742857,0.000000,0.266667
2,alm4pea3,37,1,36,0,6,0,9,9,0.243243,0.650192,0.631579,0.000000,0.166667
3,alm4pea4,36,1,35,0,5,0,9,9,0.250000,0.612159,0.670750,0.000000,0.142857
4,alm3pea1,42,15,27,0,5,1,6,7,0.166667,0.698696,0.851373,0.000000,0.185185
5,alm3pea3,39,15,24,0,4,1,3,4,0.102564,0.697605,0.810811,0.000000,0.166667
6,alm3pea4,34,1,33,0,3,0,9,9,0.264706,0.589856,0.638494,0.000000,0.090909
7,alm4pea1,30,1,29,0,2,0,10,10,0.333333,0.580221,0.579500,0.000000,0.068966
8,alm3pea2,39,15,24,0,1,0,7,7,0.179487,0.716082,0.859649,0.000000,0.041667
9,alm1pea4,27,1,26,0,0,0,5,5,0.185185,0.427422,0.339286,0.000000,0.000000


In [54]:
fig = px.bar(
    mixture_image_summary_df,
    x="source_image",
    y=["missed_peanut", "auto_false_positive", "uncertain_peanut", "uncertain_almond"],
    title="Mixtures — error and uncertainty counts by source image",
    barmode="group",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

fig = px.scatter(
    mixture_image_summary_df,
    x="auto_false_positive",
    y="missed_peanut",
    size="uncertain_total",
    color="uncertain_rate",
    hover_data=["source_image", "n_total", "n_peanut", "n_almond"],
    title="Mixture images — FN/FP/uncertainty profile",
)
fig.show()

## 16. Random-state stability

The SIMCA model uses balanced pixel sampling. Therefore, the selected conclusions must be checked across several random seeds.

This section reruns the selected final models with different random seeds and reports the variability of FN, FP and uncertainty.

In [55]:
RANDOM_STABILITY_SEEDS = list(range(10))  # Increase to range(20) for the final report.

STABILITY_MODELS = FINAL_CANDIDATE_MODELS[
    FINAL_CANDIDATE_MODELS["model_name"].isin(["B_compromise", "C_accuracy"])
].copy()

STABILITY_THREE_WAY_BASE_MODEL = "C_accuracy"

print("Seeds:", RANDOM_STABILITY_SEEDS)
display(STABILITY_MODELS)

Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,model_name,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
1,B_compromise,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,FN/FP compromise
2,C_accuracy,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1 orientation


In [56]:
def evaluate_three_way_from_object_df(
    object_df,
    set_name,
    model_name,
    low_threshold,
    high_threshold,
    seed,
):
    tmp = add_three_way_decision(
        object_df,
        score_col="peanut_pixel_ratio",
        low_threshold=float(low_threshold),
        high_threshold=float(high_threshold),
    )

    metrics = three_way_metrics(tmp)

    metrics.update({
        "model_name": model_name,
        "decision_model": "three_way_uncertain",
        "set": set_name,
        "random_state": int(seed),
        "low_threshold": float(low_threshold),
        "high_threshold": float(high_threshold),
    })

    return metrics


stability_rows = []

for seed in RANDOM_STABILITY_SEEDS:
    print(f"Random seed: {seed}")

    for _, candidate_row in STABILITY_MODELS.iterrows():
        model_name = str(candidate_row["model_name"])

        for set_info in FINAL_COMPARISON_SETS:
            metrics, object_df, projection = evaluate_hard_candidate(
                candidate_row=candidate_row,
                projection_filters=set_info["projection_filters"],
                projection_name=set_info["set"],
                train_filters=set_info["train_filters"],
                train_role=set_info["train_role"],
                random_state=int(seed),
                dilation_radius=FIXED_POSITION_DILATION_RADIUS,
            )

            metrics["stability_type"] = "hard_binary"
            stability_rows.append(metrics)

    # Three-way model with fixed thresholds selected above.
    base_row = FINAL_CANDIDATE_MODELS[
        FINAL_CANDIDATE_MODELS["model_name"].eq(STABILITY_THREE_WAY_BASE_MODEL)
    ].iloc[0]

    for set_info in FINAL_COMPARISON_SETS:
        _, base_object_df, _ = evaluate_hard_candidate(
            candidate_row=base_row,
            projection_filters=set_info["projection_filters"],
            projection_name=set_info["set"],
            train_filters=set_info["train_filters"],
            train_role=set_info["train_role"],
            random_state=int(seed),
            dilation_radius=FIXED_POSITION_DILATION_RADIUS,
        )

        tw_metrics = evaluate_three_way_from_object_df(
            object_df=base_object_df,
            set_name=set_info["set"],
            model_name="three_way_ratio",
            low_threshold=THREE_WAY_LOW,
            high_threshold=THREE_WAY_HIGH,
            seed=seed,
        )

        tw_metrics["stability_type"] = "three_way"
        stability_rows.append(tw_metrics)

stability_df = pd.DataFrame(stability_rows)

display(stability_df.head())

Random seed: 0


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Random seed: 1


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Random seed: 2


...

Random seed: 8


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


Random seed: 9


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(


,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,model_name,decision_model,set,train_role,preprocessing,n_components,rule_variant,border_width,object_threshold,random_state,truth_dilation_radius,selection_logic,stability_type,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,low_threshold,high_threshold
0,108.0,52.0,1.0,6.0,49.0,0.981132,0.890909,0.936021,0.935185,0.896552,0.936937,0.018868,0.109091,B_compromise,hard_ratio_threshold,validation,calibration_batches_1_2,absorbance_sg_d1,7.0,data_driven_emp_cv,1.0,0.80,0,3.0,FN/FP compromise,hard_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,77.0,29.0,0.0,8.0,40.0,1.000000,0.833333,0.916667,0.896104,0.783784,0.878788,0.000000,0.166667,B_compromise,hard_ratio_threshold,test,calibration_batches_1_2,absorbance_sg_d1,7.0,data_driven_emp_cv,1.0,0.80,0,3.0,FN/FP compromise,hard_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,722.0,144.0,2.0,79.0,497.0,0.986301,0.862847,0.924574,0.887812,0.645740,0.780488,0.013699,0.137153,B_compromise,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,absorbance_sg_d1,7.0,data_driven_emp_cv,1.0,0.80,0,3.0,FN/FP compromise,hard_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,108.0,51.0,2.0,4.0,51.0,0.962264,0.927273,0.944768,0.944444,0.927273,0.944444,0.037736,0.072727,C_accuracy,hard_ratio_threshold,validation,calibration_batches_1_2,absorbance_sg_d1,7.0,data_driven_emp_cv,1.0,0.85,0,3.0,max accuracy / F1 orientation,hard_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,77.0,28.0,1.0,7.0,41.0,0.965517,0.854167,0.909842,0.896104,0.800000,0.875000,0.034483,0.145833,C_accuracy,hard_ratio_threshold,test,calibration_batches_1_2,absorbance_sg_d1,7.0,data_driven_emp_cv,1.0,0.85,0,3.0,max accuracy / F1 orientation,hard_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
hard_stability_summary_df = (
    stability_df
    .query("stability_type == 'hard_binary'")
    .groupby(["model_name", "set"], as_index=False)
    .agg(
        fn_mean=("fn", "mean"),
        fn_std=("fn", "std"),
        fn_min=("fn", "min"),
        fn_max=("fn", "max"),
        fp_mean=("fp", "mean"),
        fp_std=("fp", "std"),
        fp_min=("fp", "min"),
        fp_max=("fp", "max"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        f1_mean=("f1_score", "mean"),
        f1_std=("f1_score", "std"),
    )
)

display(hard_stability_summary_df)


three_way_stability_summary_df = (
    stability_df
    .query("stability_type == 'three_way'")
    .groupby(["model_name", "set"], as_index=False)
    .agg(
        missed_peanut_mean=("missed_peanut", "mean"),
        missed_peanut_std=("missed_peanut", "std"),
        missed_peanut_min=("missed_peanut", "min"),
        missed_peanut_max=("missed_peanut", "max"),
        auto_fp_mean=("auto_false_positive", "mean"),
        auto_fp_std=("auto_false_positive", "std"),
        auto_fp_min=("auto_false_positive", "min"),
        auto_fp_max=("auto_false_positive", "max"),
        uncertain_mean=("n_uncertain", "mean"),
        uncertain_std=("n_uncertain", "std"),
        uncertain_rate_mean=("uncertain_rate", "mean"),
        uncertain_rate_std=("uncertain_rate", "std"),
    )
)

display(three_way_stability_summary_df)

,model_name,set,fn_mean,fn_std,fn_min,fn_max,fp_mean,fp_std,fp_min,fp_max,accuracy_mean,accuracy_std,f1_mean,f1_std
0,B_compromise,mixture,2.2,0.421637,2.0,3.0,77.0,5.228129,70.0,84.0,0.890305,0.006904,0.784217,0.010413
1,B_compromise,test,0.0,0.000000,0.0,0.0,8.5,1.269296,7.0,10.0,0.889610,0.016484,0.872467,0.016657
2,B_compromise,validation,0.9,0.316228,0.0,1.0,5.8,0.421637,5.0,6.0,0.937963,0.004473,0.939590,0.004279
3,C_accuracy,mixture,5.2,0.421637,5.0,6.0,56.8,4.104198,51.0,61.0,0.914127,0.005264,0.819658,0.008816
4,C_accuracy,test,0.9,0.316228,0.0,1.0,7.0,0.471405,6.0,8.0,0.897403,0.009583,0.876779,0.011186
5,C_accuracy,validation,2.0,0.000000,2.0,2.0,5.1,1.370320,2.0,6.0,0.934259,0.012688,0.935057,0.011928


,model_name,set,missed_peanut_mean,missed_peanut_std,missed_peanut_min,missed_peanut_max,auto_fp_mean,auto_fp_std,auto_fp_min,auto_fp_max,uncertain_mean,uncertain_std,uncertain_rate_mean,uncertain_rate_std
0,three_way_ratio,mixture,1.1,0.316228,1.0,2.0,36.6,3.272783,31.0,40.0,75.9,1.728840,0.105125,0.002395
1,three_way_ratio,test,0.0,0.000000,0.0,0.0,5.7,0.483046,5.0,6.0,7.1,0.875595,0.092208,0.011371
2,three_way_ratio,validation,0.0,0.000000,0.0,0.0,1.8,0.918937,1.0,4.0,11.0,1.247219,0.101852,0.011548


In [58]:
fig = px.box(
    stability_df.query("set == 'mixture' and stability_type == 'hard_binary'"),
    x="model_name",
    y="fp",
    color="model_name",
    points="all",
    title="Random-state stability — mixture FP for hard binary models",
)
fig.show()

fig = px.box(
    stability_df.query("set == 'mixture' and stability_type == 'three_way'"),
    x="model_name",
    y="auto_false_positive",
    color="model_name",
    points="all",
    title="Random-state stability — mixture automatic FP for three-way model",
)
fig.show()

fig = px.box(
    stability_df.query("set == 'mixture' and stability_type == 'three_way'"),
    x="model_name",
    y="n_uncertain",
    color="model_name",
    points="all",
    title="Random-state stability — mixture uncertain objects",
)
fig.show()

## 17. Sensitivity to mixture truth hyperparameters

The mixture pixel truth is obtained from peanut position-reference images. The dilation radius controls how permissive the peanut reference mask is.

This section checks whether FN/FP conclusions are stable when the position-reference dilation radius changes.

If results vary strongly with dilation, part of the measured error is due to uncertainty in the ground truth, not only to the SIMCA model.

In [59]:
TRUTH_DILATION_RADII = [0, 1, 2, 3, 4, 5]

TRUTH_SENSITIVITY_HARD_MODELS = FINAL_CANDIDATE_MODELS[
    FINAL_CANDIDATE_MODELS["model_name"].isin(["B_compromise", "C_accuracy"])
].copy()

display(TRUTH_SENSITIVITY_HARD_MODELS)

,model_name,preprocessing,n_components,rule_variant,border_width,object_threshold,selection_logic
1,B_compromise,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,FN/FP compromise
2,C_accuracy,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,max accuracy / F1 orientation


In [60]:
def relabel_pixel_truth_with_dilation(pixel_df, dilation_radius):
    no_truth = pixel_df.drop(
        columns=["true_peanut_pixel", "truth_available"],
        errors="ignore",
    ).copy()

    return add_pixel_truth_labels(
        pixel_df=no_truth,
        image_db=image_db,
        object_db=object_db,
        dilation_radius=int(dilation_radius),
    )


truth_sensitivity_rows = []

for dilation_radius in TRUTH_DILATION_RADII:
    print("Truth dilation radius:", dilation_radius)

    for _, candidate_row in TRUTH_SENSITIVITY_HARD_MODELS.iterrows():
        model_name = str(candidate_row["model_name"])

        # Use already-projected mixture pixels for this model, then only relabel truth.
        base_pixel_df = final_model_outputs[(model_name, "mixture")]["pixel_df"]

        pixel_df_dil = relabel_pixel_truth_with_dilation(
            pixel_df=base_pixel_df,
            dilation_radius=int(dilation_radius),
        )

        object_df_dil = aggregate_pixel_predictions_to_objects_core(
            pixel_df=pixel_df_dil,
            object_db=object_db,
            object_threshold=float(candidate_row["object_threshold"]),
            border_width=int(candidate_row["border_width"]),
            min_core_pixels=MIN_CORE_PIXELS,
            fallback_to_all_pixels=True,
        )

        metrics = binary_detection_metrics(
            object_df_dil,
            true_col="true_peanut_object",
            pred_col="predicted_peanut_object",
        )

        metrics.update({
            "model_name": model_name,
            "decision_model": "hard_ratio_threshold",
            "set": "mixture",
            "truth_dilation_radius": int(dilation_radius),
            "object_threshold": float(candidate_row["object_threshold"]),
            "border_width": int(candidate_row["border_width"]),
        })

        truth_sensitivity_rows.append(metrics)

    # Three-way truth sensitivity using the base model pixel predictions.
    base_pixel_df = final_model_outputs[(THREE_WAY_BASE_MODEL_NAME, "mixture")]["pixel_df"]

    pixel_df_dil = relabel_pixel_truth_with_dilation(
        pixel_df=base_pixel_df,
        dilation_radius=int(dilation_radius),
    )

    base_row = FINAL_CANDIDATE_MODELS[
        FINAL_CANDIDATE_MODELS["model_name"].eq(THREE_WAY_BASE_MODEL_NAME)
    ].iloc[0]

    object_df_dil = aggregate_pixel_predictions_to_objects_core(
        pixel_df=pixel_df_dil,
        object_db=object_db,
        object_threshold=float(base_row["object_threshold"]),
        border_width=int(base_row["border_width"]),
        min_core_pixels=MIN_CORE_PIXELS,
        fallback_to_all_pixels=True,
    )

    object_df_dil = add_three_way_decision(
        object_df_dil,
        score_col="peanut_pixel_ratio",
        low_threshold=THREE_WAY_LOW,
        high_threshold=THREE_WAY_HIGH,
    )

    tw_metrics = three_way_metrics(object_df_dil)

    tw_metrics.update({
        "model_name": "three_way_ratio",
        "decision_model": "three_way_uncertain",
        "set": "mixture",
        "truth_dilation_radius": int(dilation_radius),
        "low_threshold": float(THREE_WAY_LOW),
        "high_threshold": float(THREE_WAY_HIGH),
    })

    truth_sensitivity_rows.append(tw_metrics)

truth_sensitivity_df = pd.DataFrame(truth_sensitivity_rows)

display(truth_sensitivity_df)

Truth dilation radius: 0
Truth dilation radius: 1


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

Truth dilation radius: 2


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

Truth dilation radius: 3


...

Truth dilation radius: 4


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

Truth dilation radius: 5


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  ref_mask = morphology.binary_dilation(
c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\pixel_projection.py:295: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,model_name,decision_model,set,truth_dilation_radius,object_threshold,border_width,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,low_threshold,high_threshold
0,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,0,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,0,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,0,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
3,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,1,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,1,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,1,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
6,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,2,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,2,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,2,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
9,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,3,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
hard_truth_sensitivity_df = truth_sensitivity_df[
    truth_sensitivity_df["decision_model"].eq("hard_ratio_threshold")
].copy()

three_way_truth_sensitivity_df = truth_sensitivity_df[
    truth_sensitivity_df["decision_model"].eq("three_way_uncertain")
].copy()

display(
    hard_truth_sensitivity_df[
        [
            "model_name",
            "truth_dilation_radius",
            "tp", "fn", "fp", "tn",
            "fn_rate", "fp_rate",
            "accuracy", "f1_score",
            "object_threshold",
            "border_width",
        ]
    ]
)

display(
    three_way_truth_sensitivity_df[
        [
            "model_name",
            "truth_dilation_radius",
            "missed_peanut",
            "auto_false_positive",
            "uncertain_peanut",
            "uncertain_almond",
            "n_uncertain",
            "uncertain_rate",
            "low_threshold",
            "high_threshold",
        ]
    ]
)

,model_name,truth_dilation_radius,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,object_threshold,border_width
0,B_compromise,0,144.0,2.0,74.0,502.0,0.013699,0.128472,0.894737,0.791209,0.80,1.0
1,C_accuracy,0,141.0,5.0,57.0,519.0,0.034247,0.098958,0.914127,0.819767,0.85,1.0
3,B_compromise,1,144.0,2.0,74.0,502.0,0.013699,0.128472,0.894737,0.791209,0.80,1.0
4,C_accuracy,1,141.0,5.0,57.0,519.0,0.034247,0.098958,0.914127,0.819767,0.85,1.0
6,B_compromise,2,144.0,2.0,74.0,502.0,0.013699,0.128472,0.894737,0.791209,0.80,1.0
7,C_accuracy,2,141.0,5.0,57.0,519.0,0.034247,0.098958,0.914127,0.819767,0.85,1.0
9,B_compromise,3,144.0,2.0,74.0,502.0,0.013699,0.128472,0.894737,0.791209,0.80,1.0
10,C_accuracy,3,141.0,5.0,57.0,519.0,0.034247,0.098958,0.914127,0.819767,0.85,1.0
12,B_compromise,4,144.0,2.0,74.0,502.0,0.013699,0.128472,0.894737,0.791209,0.80,1.0
13,C_accuracy,4,141.0,5.0,57.0,519.0,0.034247,0.098958,0.914127,0.819767,0.85,1.0


,model_name,truth_dilation_radius,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,n_uncertain,uncertain_rate,low_threshold,high_threshold
2,three_way_ratio,0,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9
5,three_way_ratio,1,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9
8,three_way_ratio,2,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9
11,three_way_ratio,3,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9
14,three_way_ratio,4,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9
17,three_way_ratio,5,1.0,34.0,9.0,66.0,75.0,0.103878,0.7,0.9


In [62]:
fig = px.line(
    hard_truth_sensitivity_df,
    x="truth_dilation_radius",
    y="fp",
    color="model_name",
    markers=True,
    title="Truth sensitivity — mixture FP for hard binary models",
)
fig.show()

fig = px.line(
    hard_truth_sensitivity_df,
    x="truth_dilation_radius",
    y="fn",
    color="model_name",
    markers=True,
    title="Truth sensitivity — mixture FN for hard binary models",
)
fig.show()

fig = px.line(
    three_way_truth_sensitivity_df,
    x="truth_dilation_radius",
    y=["missed_peanut", "auto_false_positive", "n_uncertain"],
    markers=True,
    title="Truth sensitivity — three-way model",
)
fig.show()

## 18. Final pre-error-modelling summary

This cell summarizes what should be checked before moving to explicit error modelling.

In [63]:
print("=== Final hard binary comparison ===")
display(
    final_hard_comparison_df[FINAL_COMPARE_COLS]
    .sort_values(["set", "fn", "fp", "f1_score", "accuracy"],
                 ascending=[True, True, True, False, False])
)

print("=== Three-way comparison ===")
display(three_way_results_df)

print("=== Mixture image summary ===")
display(mixture_image_summary_df)

print("=== Random-state stability summary: hard models ===")
display(hard_stability_summary_df)

print("=== Random-state stability summary: three-way model ===")
display(three_way_stability_summary_df)

print("=== Truth dilation sensitivity: hard models ===")
display(hard_truth_sensitivity_df)

print("=== Truth dilation sensitivity: three-way model ===")
display(three_way_truth_sensitivity_df)

=== Final hard binary comparison ===


,model_name,decision_model,set,train_role,tp,fn,fp,tn,fn_rate,fp_rate,peanut_sensitivity,almond_specificity,accuracy,f1_score,balanced_accuracy,preprocessing,n_components,rule_variant,border_width,object_threshold,truth_dilation_radius,selection_logic
5,B_compromise,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,144,2,74,502,0.013699,0.128472,0.986301,0.871528,0.894737,0.791209,0.928915,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
2,A_safety_FN0,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,144,2,88,488,0.013699,0.152778,0.986301,0.847222,0.875346,0.761905,0.916762,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
8,C_accuracy,hard_ratio_threshold,mixture,deployment_pure_batches_1_2_3_4,141,5,57,519,0.034247,0.098958,0.965753,0.901042,0.914127,0.819767,0.933398,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation
7,C_accuracy,hard_ratio_threshold,test,calibration_batches_1_2,29,0,6,42,0.000000,0.125000,1.000000,0.875000,0.922078,0.906250,0.937500,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation
4,B_compromise,hard_ratio_threshold,test,calibration_batches_1_2,29,0,7,41,0.000000,0.145833,1.000000,0.854167,0.909091,0.892308,0.927083,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
1,A_safety_FN0,hard_ratio_threshold,test,calibration_batches_1_2,29,0,9,39,0.000000,0.187500,1.000000,0.812500,0.883117,0.865672,0.906250,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
0,A_safety_FN0,hard_ratio_threshold,validation,calibration_batches_1_2,53,0,6,49,0.000000,0.109091,1.000000,0.890909,0.944444,0.946429,0.945455,absorbance_sg_d1,7,data_driven_emp_cv,1,0.75,3,"FN=0 orientation, then min FP"
3,B_compromise,hard_ratio_threshold,validation,calibration_batches_1_2,52,1,4,51,0.018868,0.072727,0.981132,0.927273,0.953704,0.954128,0.954202,absorbance_sg_d1,7,data_driven_emp_cv,1,0.80,3,FN/FP compromise
6,C_accuracy,hard_ratio_threshold,validation,calibration_batches_1_2,51,2,2,53,0.037736,0.036364,0.962264,0.963636,0.962963,0.962264,0.962950,absorbance_sg_d1,7,data_driven_emp_cv,1,0.85,3,max accuracy / F1 orientation


=== Three-way comparison ===


,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,model_name,decision_model,set,score_col,low_threshold,high_threshold,base_model
0,108,99,9,0.083333,53,55,0,1,4,5,0.000000,0.018182,0.075472,0.090909,99,49,0,1,49,1.000000,0.980000,0.990000,0.989899,0.980000,0.989899,0.000000,0.020000,three_way_ratio,three_way_uncertain,validation,peanut_pixel_ratio,0.7,0.9,C_accuracy
1,77,70,7,0.090909,29,48,0,5,1,6,0.000000,0.104167,0.034483,0.125000,70,28,0,5,37,1.000000,0.880952,0.940476,0.928571,0.848485,0.918033,0.000000,0.119048,three_way_ratio,three_way_uncertain,test,peanut_pixel_ratio,0.7,0.9,C_accuracy
2,722,647,75,0.103878,146,576,1,34,9,66,0.006849,0.059028,0.061644,0.114583,647,136,1,34,476,0.992701,0.933333,0.963017,0.945904,0.800000,0.885993,0.007299,0.066667,three_way_ratio,three_way_uncertain,mixture,peanut_pixel_ratio,0.7,0.9,C_accuracy


=== Mixture image summary ===


...

,model_name,set,missed_peanut_mean,missed_peanut_std,missed_peanut_min,missed_peanut_max,auto_fp_mean,auto_fp_std,auto_fp_min,auto_fp_max,uncertain_mean,uncertain_std,uncertain_rate_mean,uncertain_rate_std
0,three_way_ratio,mixture,1.1,0.316228,1.0,2.0,36.6,3.272783,31.0,40.0,75.9,1.728840,0.105125,0.002395
1,three_way_ratio,test,0.0,0.000000,0.0,0.0,5.7,0.483046,5.0,6.0,7.1,0.875595,0.092208,0.011371
2,three_way_ratio,validation,0.0,0.000000,0.0,0.0,1.8,0.918937,1.0,4.0,11.0,1.247219,0.101852,0.011548


=== Truth dilation sensitivity: hard models ===


,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,model_name,decision_model,set,truth_dilation_radius,object_threshold,border_width,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,low_threshold,high_threshold
0,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,0,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,0,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,1,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,1,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,2,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,2,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,3,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,3,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,722.0,144.0,2.0,74.0,502.0,0.986301,0.871528,0.928915,0.894737,0.660550,0.791209,0.013699,0.128472,B_compromise,hard_ratio_threshold,mixture,4,0.80,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,722.0,141.0,5.0,57.0,519.0,0.965753,0.901042,0.933398,0.914127,0.712121,0.819767,0.034247,0.098958,C_accuracy,hard_ratio_threshold,mixture,4,0.85,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


=== Truth dilation sensitivity: three-way model ===


,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,model_name,decision_model,set,truth_dilation_radius,object_threshold,border_width,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,low_threshold,high_threshold
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,0,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,1,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,2,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,3,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,4,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,three_way_ratio,three_way_uncertain,mixture,5,NaN,NaN,722.0,647.0,75.0,0.103878,146.0,576.0,1.0,34.0,9.0,66.0,0.006849,0.059028,0.061644,0.114583,647.0,136.0,1.0,34.0,476.0,0.992701,0.933333,0.963017,0.945904,0.8,0.885993,0.007299,0.066667,0.7,0.9


# Sum up

## Model A: peanut sensibility
- "model_name": "A_safety_FN0",
- "preprocessing": "absorbance_sg_d1",
- "n_components": 7,
- "rule_variant": "data_driven_emp_cv",
- "border_width": 1,
- "object_threshold": 0.75,
- "selection_logic": "FN=0, then min FP",
- => mixture: 2 FN; 88 FP; 

## Model B: introduces FN
- "model_name": "B_compromise",
- "preprocessing": "absorbance_sg_d1",
- "n_components": 7,
- "rule_variant": "data_driven_emp_cv",
- "border_width": 1,
- "object_threshold": 0.80,
- "selection_logic": "cost compromise",
- => mixture: 2 FN; 74 FP; 

## Model C: best accuracy
- "model_name": "C_accuracy",
- "preprocessing": "absorbance_sg_d1",
- "n_components": 7,
- "rule_variant": "data_driven_emp_cv",
- "border_width": 1,
- "object_threshold": 0.85,
- "selection_logic": "max accuracy / F1",
- => mixture: 5 FN; 57 FP; 

## Object-score:
- lambda 2: 14 FN; 26 FP
- lambda 5: 1 FN; 85 FP 
- lambda 10: 1 FN; 85 FP 


We keep B, C and Object-score with lambda 5/10

In [64]:
# Si ta table mixture s'appelle autrement, remplace mixture_score_df par le bon nom.
# Par exemple : mixture_score_df = mixture_score_df.copy()
# ou mixture_score_df = mixture_object_score_df.copy()

simple_logistic_results_df, simple_logistic_models = evaluate_simple_object_score_models(
    validation_score_df=validation_score_df,
    test_score_df=test_score_df,
    mixture_score_df=mixture_score_df if "mixture_score_df" in globals() else None,
    feature_sets=SIMPLE_OBJECT_SCORE_FEATURE_SETS,
    C_values=LOGISTIC_C_VALUES,
    lambda_values=LAMBDA_FN_VALUES,
    random_state=RANDOM_STATE,
)

display(
    simple_logistic_results_df
    .sort_values(
        ["set", "lambda_fn", "fn", "fp", "f1_score", "accuracy"],
        ascending=[True, True, True, True, False, False],
    )
    .reset_index(drop=True)
)

,n,tp,fn,fp,tn,peanut_sensitivity,almond_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,feature_set,features,C,lambda_fn,prob_threshold,set,decision_model
0,722,146,0,99,477,1.000000,0.828125,0.914062,0.862881,0.595918,0.746803,0.000000,0.171875,ratio_plus_margin_mean,"logit_peanut_ratio, margin_norm_mean",0.30,2.0,0.37,mixture,simple_logistic_object_score
1,722,146,0,100,476,1.000000,0.826389,0.913194,0.861496,0.593496,0.744898,0.000000,0.173611,ratio_plus_margin_mean,"logit_peanut_ratio, margin_norm_mean",1.00,2.0,0.27,mixture,simple_logistic_object_score
2,722,145,1,71,505,0.993151,0.876736,0.934943,0.900277,0.671296,0.801105,0.006849,0.123264,current_full,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.30,2.0,0.36,mixture,simple_logistic_object_score
3,722,145,1,92,484,0.993151,0.840278,0.916714,0.871191,0.611814,0.757180,0.006849,0.159722,compact_stat_margin,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.01,2.0,0.51,mixture,simple_logistic_object_score
4,722,145,1,97,479,0.993151,0.831597,0.912374,0.864266,0.599174,0.747423,0.006849,0.168403,ratio_plus_margin_mean,"logit_peanut_ratio, margin_norm_mean",0.10,2.0,0.44,mixture,simple_logistic_object_score
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,108,53,0,6,49,1.000000,0.890909,0.945455,0.944444,0.898305,0.946429,0.000000,0.109091,compact_stat_margin,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.30,10.0,0.37,validation_train,simple_logistic_object_score
266,108,53,0,6,49,1.000000,0.890909,0.945455,0.944444,0.898305,0.946429,0.000000,0.109091,compact_stat_margin,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.10,10.0,0.45,validation_train,simple_logistic_object_score
267,108,53,0,6,49,1.000000,0.890909,0.945455,0.944444,0.898305,0.946429,0.000000,0.109091,compact_stat_margin,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.03,10.0,0.50,validation_train,simple_logistic_object_score
268,108,53,0,6,49,1.000000,0.890909,0.945455,0.944444,0.898305,0.946429,0.000000,0.109091,compact_stat_margin,"logit_peanut_ratio, stat_norm_median, stat_nor...",0.01,10.0,0.51,validation_train,simple_logistic_object_score


In [65]:
simple_logistic_results_df[simple_logistic_results_df['set']=='mixture']['feature_set'].value_counts()

feature_set
ratio_only                15
ratio_plus_margin_mean    15
ratio_plus_stat_median    15
compact_margin            15
compact_stat_margin       15
current_full              15
Name: count, dtype: int64

In [66]:
mask_1 = ((simple_logistic_results_df['set']=='mixture') & (simple_logistic_results_df['lambda_fn']!=2.0))
cols = ["feature_set","C","lambda_fn","prob_threshold","tp", "fn", "fp", "tn","fn_rate", "fp_rate","accuracy", "f1_score","peanut_sensitivity", "almond_specificity"]

In [67]:
display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'current_full')].sort_values(
    ["fn", "fp", "f1_score", "accuracy"], ascending=[True, True, False, False]
)[cols])

display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'ratio_only')].sort_values(
    ["lambda_fn", "C","fn", "fp", "f1_score", "accuracy"], ascending=[True, True, True, True, False, False]
)[cols])

display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'ratio_plus_margin_mean')].sort_values(
    ["lambda_fn", "C","fn", "fp", "f1_score", "accuracy"], ascending=[True, True, True, True, False, False]
)[cols])

display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'ratio_plus_stat_median')].sort_values(
    ["lambda_fn", "C","fn", "fp", "f1_score", "accuracy"], ascending=[True, True, True, True, False, False]
)[cols])

display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'compact_margin')].sort_values(
    ["lambda_fn", "C","fn", "fp", "f1_score", "accuracy"], ascending=[True, True, True, True, False, False]
)[cols])

display(simple_logistic_results_df[mask_1 & (simple_logistic_results_df['feature_set'] == 'compact_stat_margin')].sort_values(
    ["lambda_fn", "C","fn", "fp", "f1_score", "accuracy"], ascending=[True, True, True, True, False, False]
)[cols])

,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
266,current_full,0.01,5.0,0.49,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
269,current_full,0.01,10.0,0.49,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
239,current_full,0.30,5.0,0.36,145,1,71,505,0.006849,0.123264,0.900277,0.801105,0.993151,0.876736
242,current_full,0.30,10.0,0.36,145,1,71,505,0.006849,0.123264,0.900277,0.801105,0.993151,0.876736
257,current_full,0.03,5.0,0.48,145,1,75,501,0.006849,0.130208,0.894737,0.792350,0.993151,0.869792
260,current_full,0.03,10.0,0.48,145,1,75,501,0.006849,0.130208,0.894737,0.792350,0.993151,0.869792
230,current_full,1.00,5.0,0.19,145,1,85,491,0.006849,0.147569,0.880886,0.771277,0.993151,0.852431
233,current_full,1.00,10.0,0.19,145,1,85,491,0.006849,0.147569,0.880886,0.771277,0.993151,0.852431
248,current_full,0.10,5.0,0.37,145,1,89,487,0.006849,0.154514,0.875346,0.763158,0.993151,0.845486
251,current_full,0.10,10.0,0.37,145,1,89,487,0.006849,0.154514,0.875346,0.763158,0.993151,0.845486


,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
41,ratio_only,0.01,5.0,0.48,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
32,ratio_only,0.03,5.0,0.46,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
23,ratio_only,0.10,5.0,0.42,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
14,ratio_only,0.30,5.0,0.36,145,1,102,474,0.006849,0.177083,0.857341,0.737913,0.993151,0.822917
5,ratio_only,1.00,5.0,0.28,145,1,102,474,0.006849,0.177083,0.857341,0.737913,0.993151,0.822917
44,ratio_only,0.01,10.0,0.48,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
35,ratio_only,0.03,10.0,0.46,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
26,ratio_only,0.10,10.0,0.42,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
17,ratio_only,0.30,10.0,0.36,145,1,102,474,0.006849,0.177083,0.857341,0.737913,0.993151,0.822917
8,ratio_only,1.00,10.0,0.28,145,1,102,474,0.006849,0.177083,0.857341,0.737913,0.993151,0.822917


,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
86,ratio_plus_margin_mean,0.01,5.0,0.49,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
77,ratio_plus_margin_mean,0.03,5.0,0.48,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
68,ratio_plus_margin_mean,0.10,5.0,0.44,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
59,ratio_plus_margin_mean,0.30,5.0,0.37,146,0,99,477,0.000000,0.171875,0.862881,0.746803,1.000000,0.828125
50,ratio_plus_margin_mean,1.00,5.0,0.27,146,0,100,476,0.000000,0.173611,0.861496,0.744898,1.000000,0.826389
89,ratio_plus_margin_mean,0.01,10.0,0.49,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
80,ratio_plus_margin_mean,0.03,10.0,0.48,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
71,ratio_plus_margin_mean,0.10,10.0,0.44,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
62,ratio_plus_margin_mean,0.30,10.0,0.37,146,0,99,477,0.000000,0.171875,0.862881,0.746803,1.000000,0.828125
53,ratio_plus_margin_mean,1.00,10.0,0.27,146,0,100,476,0.000000,0.173611,0.861496,0.744898,1.000000,0.826389


,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
131,ratio_plus_stat_median,0.01,5.0,0.50,146,0,91,485,0.0,0.157986,0.873961,0.762402,1.0,0.842014
122,ratio_plus_stat_median,0.03,5.0,0.48,146,0,96,480,0.0,0.166667,0.867036,0.752577,1.0,0.833333
113,ratio_plus_stat_median,0.10,5.0,0.44,146,0,97,479,0.0,0.168403,0.865651,0.750643,1.0,0.831597
104,ratio_plus_stat_median,0.30,5.0,0.37,146,0,98,478,0.0,0.170139,0.864266,0.748718,1.0,0.829861
95,ratio_plus_stat_median,1.00,5.0,0.27,146,0,98,478,0.0,0.170139,0.864266,0.748718,1.0,0.829861
134,ratio_plus_stat_median,0.01,10.0,0.50,146,0,91,485,0.0,0.157986,0.873961,0.762402,1.0,0.842014
125,ratio_plus_stat_median,0.03,10.0,0.48,146,0,96,480,0.0,0.166667,0.867036,0.752577,1.0,0.833333
116,ratio_plus_stat_median,0.10,10.0,0.44,146,0,97,479,0.0,0.168403,0.865651,0.750643,1.0,0.831597
107,ratio_plus_stat_median,0.30,10.0,0.37,146,0,98,478,0.0,0.170139,0.864266,0.748718,1.0,0.829861
98,ratio_plus_stat_median,1.00,10.0,0.27,146,0,98,478,0.0,0.170139,0.864266,0.748718,1.0,0.829861


,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
176,compact_margin,0.01,5.0,0.50,145,1,98,478,0.006849,0.170139,0.862881,0.745501,0.993151,0.829861
167,compact_margin,0.03,5.0,0.49,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
158,compact_margin,0.10,5.0,0.44,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
149,compact_margin,0.30,5.0,0.36,145,1,100,476,0.006849,0.173611,0.860111,0.741688,0.993151,0.826389
140,compact_margin,1.00,5.0,0.26,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
179,compact_margin,0.01,10.0,0.50,145,1,98,478,0.006849,0.170139,0.862881,0.745501,0.993151,0.829861
170,compact_margin,0.03,10.0,0.49,145,1,97,479,0.006849,0.168403,0.864266,0.747423,0.993151,0.831597
161,compact_margin,0.10,10.0,0.44,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125
152,compact_margin,0.30,10.0,0.36,145,1,100,476,0.006849,0.173611,0.860111,0.741688,0.993151,0.826389
143,compact_margin,1.00,10.0,0.26,145,1,99,477,0.006849,0.171875,0.861496,0.743590,0.993151,0.828125


,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
221,compact_stat_margin,0.01,5.0,0.51,145,1,92,484,0.006849,0.159722,0.871191,0.757180,0.993151,0.840278
212,compact_stat_margin,0.03,5.0,0.50,145,1,92,484,0.006849,0.159722,0.871191,0.757180,0.993151,0.840278
203,compact_stat_margin,0.10,5.0,0.45,145,1,95,481,0.006849,0.164931,0.867036,0.751295,0.993151,0.835069
194,compact_stat_margin,0.30,5.0,0.37,145,1,95,481,0.006849,0.164931,0.867036,0.751295,0.993151,0.835069
185,compact_stat_margin,1.00,5.0,0.25,146,0,95,481,0.000000,0.164931,0.868421,0.754522,1.000000,0.835069
224,compact_stat_margin,0.01,10.0,0.51,145,1,92,484,0.006849,0.159722,0.871191,0.757180,0.993151,0.840278
215,compact_stat_margin,0.03,10.0,0.50,145,1,92,484,0.006849,0.159722,0.871191,0.757180,0.993151,0.840278
206,compact_stat_margin,0.10,10.0,0.45,145,1,95,481,0.006849,0.164931,0.867036,0.751295,0.993151,0.835069
197,compact_stat_margin,0.30,10.0,0.37,145,1,95,481,0.006849,0.164931,0.867036,0.751295,0.993151,0.835069
188,compact_stat_margin,1.00,10.0,0.25,146,0,95,481,0.000000,0.164931,0.868421,0.754522,1.000000,0.835069


In [68]:
if "mixture" in simple_logistic_results_df["set"].unique():
    display(
        simple_logistic_results_df
        .query("set == 'mixture'")
        .sort_values(
            ["fn", "fp", "f1_score", "accuracy"],
            ascending=[ True, True, False, False],
        )
        [
            [
                "feature_set",
                "C",
                "lambda_fn",
                "prob_threshold",
                "tp", "fn", "fp", "tn",
                "fn_rate", "fp_rate",
                "accuracy", "f1_score",
                "peanut_sensitivity", "almond_specificity",
            ]
        ]
        .reset_index(drop=True)
    )

,feature_set,C,lambda_fn,prob_threshold,tp,fn,fp,tn,fn_rate,fp_rate,accuracy,f1_score,peanut_sensitivity,almond_specificity
0,ratio_plus_stat_median,0.01,5.0,0.50,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
1,ratio_plus_stat_median,0.01,10.0,0.50,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
2,current_full,0.01,5.0,0.49,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
3,current_full,0.01,10.0,0.49,146,0,91,485,0.000000,0.157986,0.873961,0.762402,1.000000,0.842014
4,compact_stat_margin,1.00,5.0,0.25,146,0,95,481,0.000000,0.164931,0.868421,0.754522,1.000000,0.835069
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,current_full,0.01,2.0,0.59,136,10,30,546,0.068493,0.052083,0.944598,0.871795,0.931507,0.947917
86,compact_margin,1.00,2.0,0.69,136,10,37,539,0.068493,0.064236,0.934903,0.852665,0.931507,0.935764
87,current_full,0.03,2.0,0.64,135,11,26,550,0.075342,0.045139,0.948753,0.879479,0.924658,0.954861
88,current_full,1.00,2.0,0.73,132,14,26,550,0.095890,0.045139,0.944598,0.868421,0.904110,0.954861


# Introducing 'uncertain' category

In [69]:
def add_three_way_decision(
    df,
    score_col="peanut_pixel_ratio",
    low_threshold=0.70,
    high_threshold=0.85,
    output_col="three_way_decision",
):
    if low_threshold >= high_threshold:
        raise ValueError("low_threshold must be < high_threshold.")

    out = df.copy()
    score = out[score_col].astype(float)

    out[output_col] = np.select(
        [
            score < float(low_threshold),
            score >= float(high_threshold),
        ],
        [
            "non_peanut",
            "peanut",
        ],
        default="uncertain",
    )

    return out


def three_way_metrics(
    df,
    decision_col="three_way_decision",
    true_col="true_peanut_object",
):
    d = df.dropna(subset=[decision_col, true_col]).copy()
    y_true = d[true_col].astype(bool)

    is_peanut_pred = d[decision_col].eq("peanut")
    is_non_peanut_pred = d[decision_col].eq("non_peanut")
    is_uncertain = d[decision_col].eq("uncertain")

    n = len(d)
    n_uncertain = int(is_uncertain.sum())
    uncertain_rate = n_uncertain / n if n > 0 else np.nan

    # Automatic decided subset only
    decided = d[~is_uncertain].copy()

    if len(decided) > 0:
        decided["predicted_peanut_object"] = decided[decision_col].eq("peanut")
        decided_metrics = binary_detection_metrics(
            decided,
            true_col=true_col,
            pred_col="predicted_peanut_object",
        )
    else:
        decided_metrics = {
            "n": 0,
            "tp": 0,
            "fn": 0,
            "fp": 0,
            "tn": 0,
            "peanut_sensitivity": np.nan,
            "almond_specificity": np.nan,
            "balanced_accuracy": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "f1_score": np.nan,
            "fn_rate": np.nan,
            "fp_rate": np.nan,
        }

    # Operational safety view:
    # uncertain is not automatically accepted as non-peanut.
    # A peanut is missed only if it is classified as non_peanut.
    missed_peanut = y_true & is_non_peanut_pred
    auto_false_positive = (~y_true) & is_peanut_pred
    review_almond = (~y_true) & is_uncertain
    review_peanut = y_true & is_uncertain

    n_peanut = int(y_true.sum())
    n_almond = int((~y_true).sum())

    out = {
        "n_total": int(n),
        "n_decided": int(len(decided)),
        "n_uncertain": int(n_uncertain),
        "uncertain_rate": float(uncertain_rate),

        "n_peanut": n_peanut,
        "n_almond": n_almond,

        "missed_peanut": int(missed_peanut.sum()),
        "auto_false_positive": int(auto_false_positive.sum()),
        "uncertain_peanut": int(review_peanut.sum()),
        "uncertain_almond": int(review_almond.sum()),

        "missed_peanut_rate": (
            float(missed_peanut.sum() / n_peanut)
            if n_peanut > 0 else np.nan
        ),
        "auto_fp_rate": (
            float(auto_false_positive.sum() / n_almond)
            if n_almond > 0 else np.nan
        ),
        "peanut_uncertain_rate": (
            float(review_peanut.sum() / n_peanut)
            if n_peanut > 0 else np.nan
        ),
        "almond_uncertain_rate": (
            float(review_almond.sum() / n_almond)
            if n_almond > 0 else np.nan
        ),
    }

    for key, value in decided_metrics.items():
        out[f"decided_{key}"] = value

    return out

In [70]:
def three_way_threshold_grid(
    df,
    score_col="peanut_pixel_ratio",
    true_col="true_peanut_object",
    low_values=np.arange(0.50, 0.81, 0.05),
    high_values=np.arange(0.70, 0.96, 0.05),
    lambda_fn=5.0,
    review_penalty=0.20,
):
    rows = []

    for low in low_values:
        for high in high_values:
            if low >= high:
                continue

            tmp = add_three_way_decision(
                df,
                score_col=score_col,
                low_threshold=float(low),
                high_threshold=float(high),
            )

            metrics = three_way_metrics(
                tmp,
                decision_col="three_way_decision",
                true_col=true_col,
            )

            # Cost with three terms:
            # - missed peanuts are the most costly
            # - automatic false positives are costly
            # - uncertain objects are not errors, but they generate manual review
            cost = (
                lambda_fn * metrics["missed_peanut_rate"]
                + metrics["auto_fp_rate"]
                + review_penalty * metrics["uncertain_rate"]
            )

            metrics.update({
                "score_col": score_col,
                "low_threshold": float(low),
                "high_threshold": float(high),
                "lambda_fn": float(lambda_fn),
                "review_penalty": float(review_penalty),
                "three_way_cost": float(cost),
            })

            rows.append(metrics)

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "three_way_cost",
                "missed_peanut",
                "auto_false_positive",
                "uncertain_rate",
            ],
            ascending=[True, True, True, True],
        )
        .reset_index(drop=True)
    )

In [71]:
three_way_grid_validation_ratio = three_way_threshold_grid(
    validation_score_df,
    score_col="peanut_pixel_ratio",
    low_values=np.arange(0.50, 0.86, 0.05),
    high_values=np.arange(0.65, 0.96, 0.05),
    lambda_fn=5.0,
    review_penalty=0.20,
)

display(three_way_grid_validation_ratio.head(20))

best_three_way_ratio = three_way_grid_validation_ratio.iloc[0]

RATIO_LOW = float(best_three_way_ratio["low_threshold"])
RATIO_HIGH = float(best_three_way_ratio["high_threshold"])

print("Selected ratio thresholds:", RATIO_LOW, RATIO_HIGH)

,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,score_col,low_threshold,high_threshold,lambda_fn,review_penalty,three_way_cost
0,108,99,9,0.083333,53,55,0,1,4,5,0.0,0.018182,0.075472,0.090909,99,49,0,1,49,1.0,0.980000,0.990000,0.989899,0.980000,0.989899,0.0,0.020000,peanut_pixel_ratio,0.70,0.90,5.0,0.2,0.034848
1,108,97,11,0.101852,53,55,0,1,4,7,0.0,0.018182,0.075472,0.127273,97,49,0,1,47,1.0,0.979167,0.989583,0.989691,0.980000,0.989899,0.0,0.020833,peanut_pixel_ratio,0.65,0.90,5.0,0.2,0.038552
2,108,86,22,0.203704,53,55,0,0,16,6,0.0,0.000000,0.301887,0.109091,86,37,0,0,49,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.70,0.95,5.0,0.2,0.040741
3,108,94,14,0.129630,53,55,0,1,4,10,0.0,0.018182,0.075472,0.181818,94,49,0,1,44,1.0,0.977778,0.988889,0.989362,0.980000,0.989899,0.0,0.022222,peanut_pixel_ratio,0.60,0.90,5.0,0.2,0.044108
4,108,84,24,0.222222,53,55,0,0,16,8,0.0,0.000000,0.301887,0.145455,84,37,0,0,47,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.65,0.95,5.0,0.2,0.044444
5,108,93,15,0.138889,53,55,0,1,4,11,0.0,0.018182,0.075472,0.200000,93,49,0,1,43,1.0,0.977273,0.988636,0.989247,0.980000,0.989899,0.0,0.022727,peanut_pixel_ratio,0.55,0.90,5.0,0.2,0.045960
6,108,102,6,0.055556,53,55,0,2,2,4,0.0,0.036364,0.037736,0.072727,102,51,0,2,49,1.0,0.960784,0.980392,0.980392,0.962264,0.980769,0.0,0.039216,peanut_pixel_ratio,0.70,0.85,5.0,0.2,0.047475
7,108,91,17,0.157407,53,55,0,1,4,13,0.0,0.018182,0.075472,0.236364,91,49,0,1,41,1.0,0.976190,0.988095,0.989011,0.980000,0.989899,0.0,0.023810,peanut_pixel_ratio,0.50,0.90,5.0,0.2,0.049663
8,108,81,27,0.250000,53,55,0,0,16,11,0.0,0.000000,0.301887,0.200000,81,37,0,0,44,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,peanut_pixel_ratio,0.60,0.95,5.0,0.2,0.050000
9,108,100,8,0.074074,53,55,0,2,2,6,0.0,0.036364,0.037736,0.109091,100,51,0,2,47,1.0,0.959184,0.979592,0.980000,0.962264,0.980769,0.0,0.040816,peanut_pixel_ratio,0.65,0.85,5.0,0.2,0.051178


Selected ratio thresholds: 0.7000000000000002 0.9000000000000002


In [72]:
if "p_peanut_object" in validation_score_df.columns:
    three_way_grid_validation_prob = three_way_threshold_grid(
        validation_score_df,
        score_col="p_peanut_object",
        low_values=np.arange(0.05, 0.51, 0.05),
        high_values=np.arange(0.20, 0.96, 0.05),
        lambda_fn=5.0,
        review_penalty=0.20,
    )

    display(three_way_grid_validation_prob.head(20))

    best_three_way_prob = three_way_grid_validation_prob.iloc[0]

    PROB_LOW = float(best_three_way_prob["low_threshold"])
    PROB_HIGH = float(best_three_way_prob["high_threshold"])

    print("Selected probability thresholds:", PROB_LOW, PROB_HIGH)

,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,score_col,low_threshold,high_threshold,lambda_fn,review_penalty,three_way_cost
0,108,101,7,0.064815,53,55,0,0,2,5,0.0,0.000000,0.037736,0.090909,101,51,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.20,0.75,5.0,0.2,0.012963
1,108,101,7,0.064815,53,55,0,0,2,5,0.0,0.000000,0.037736,0.090909,101,51,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.25,0.75,5.0,0.2,0.012963
2,108,100,8,0.074074,53,55,0,0,3,5,0.0,0.000000,0.056604,0.090909,100,50,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.20,0.80,5.0,0.2,0.014815
3,108,100,8,0.074074,53,55,0,0,3,5,0.0,0.000000,0.056604,0.090909,100,50,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.25,0.80,5.0,0.2,0.014815
4,108,99,9,0.083333,53,55,0,0,2,7,0.0,0.000000,0.037736,0.127273,99,51,0,0,48,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.15,0.75,5.0,0.2,0.016667
5,108,98,10,0.092593,53,55,0,0,3,7,0.0,0.000000,0.056604,0.127273,98,50,0,0,48,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.15,0.80,5.0,0.2,0.018519
6,108,98,10,0.092593,53,55,0,0,5,5,0.0,0.000000,0.094340,0.090909,98,48,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.20,0.85,5.0,0.2,0.018519
7,108,98,10,0.092593,53,55,0,0,5,5,0.0,0.000000,0.094340,0.090909,98,48,0,0,50,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.25,0.85,5.0,0.2,0.018519
8,108,97,11,0.101852,53,55,0,0,2,9,0.0,0.000000,0.037736,0.163636,97,51,0,0,46,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.10,0.75,5.0,0.2,0.020370
9,108,96,12,0.111111,53,55,0,0,3,9,0.0,0.000000,0.056604,0.163636,96,50,0,0,46,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.000000,p_peanut_object,0.10,0.80,5.0,0.2,0.022222


Selected probability thresholds: 0.2 0.7499999999999998


In [73]:
def evaluate_three_way_on_sets(
    sets,
    score_col,
    low_threshold,
    high_threshold,
    model_name,
):
    rows = []
    tables = {}

    for set_name, df in sets.items():
        tmp = add_three_way_decision(
            df,
            score_col=score_col,
            low_threshold=low_threshold,
            high_threshold=high_threshold,
        )

        metrics = three_way_metrics(tmp)

        metrics.update({
            "set": set_name,
            "model_name": model_name,
            "score_col": score_col,
            "low_threshold": float(low_threshold),
            "high_threshold": float(high_threshold),
        })

        rows.append(metrics)
        tables[set_name] = tmp

    return pd.DataFrame(rows), tables


score_sets = {
    "validation": validation_score_df,
    "test": test_score_df,
}

if "mixture_score_df" in globals():
    score_sets["mixture"] = mixture_score_df


three_way_ratio_results_df, three_way_ratio_tables = evaluate_three_way_on_sets(
    sets=score_sets,
    score_col="peanut_pixel_ratio",
    low_threshold=RATIO_LOW,
    high_threshold=RATIO_HIGH,
    model_name="three_way_ratio",
)

display(three_way_ratio_results_df)

,n_total,n_decided,n_uncertain,uncertain_rate,n_peanut,n_almond,missed_peanut,auto_false_positive,uncertain_peanut,uncertain_almond,missed_peanut_rate,auto_fp_rate,peanut_uncertain_rate,almond_uncertain_rate,decided_n,decided_tp,decided_fn,decided_fp,decided_tn,decided_peanut_sensitivity,decided_almond_specificity,decided_balanced_accuracy,decided_accuracy,decided_precision,decided_f1_score,decided_fn_rate,decided_fp_rate,set,model_name,score_col,low_threshold,high_threshold
0,108,99,9,0.083333,53,55,0,1,4,5,0.000000,0.018182,0.075472,0.090909,99,49,0,1,49,1.000000,0.980000,0.990000,0.989899,0.980000,0.989899,0.000000,0.020000,validation,three_way_ratio,peanut_pixel_ratio,0.7,0.9
1,77,71,6,0.077922,29,48,0,5,1,5,0.000000,0.104167,0.034483,0.104167,71,28,0,5,38,1.000000,0.883721,0.941860,0.929577,0.848485,0.918033,0.000000,0.116279,test,three_way_ratio,peanut_pixel_ratio,0.7,0.9
2,722,648,74,0.102493,146,576,1,34,9,65,0.006849,0.059028,0.061644,0.112847,648,136,1,34,477,0.992701,0.933464,0.963082,0.945988,0.800000,0.885993,0.007299,0.066536,mixture,three_way_ratio,peanut_pixel_ratio,0.7,0.9
